# Functional Analysis of Pediatric Microbiome Data

This notebook performs comprehensive analysis of microbiome data from kids cohorts, focusing on functional annotation done by PICRUST 2 of a 16S metagenome.

## Dataset Description:
- **ASV_table.tsv**: Relative abundance of taxa (ASV_ID) across samples
- **ASV_tax_species.user.tsv**: Taxonomic classification of ASVs
- **EC_pred_metagenome_unstrat_descrip 1.tsv**: Enzyme Commission (EC) codes and predicted abundances
- **KO_pred_metagenome_unstrat_descrip 1.tsv**: KEGG Orthology (KO) pathways and abundances
- **METACYC_path_abun_unstrat_descrip 1.tsv**: MetaCyc pathway abundances
- **data_formicrobiome02042025ok.xlsx**: Metadata including nutrition, infection, and dietary patterns

## Analysis Goals:
- Load and standardize all datasets
- Create data dictionary and quality control reports
- Analyze relationships between microbiome function, nutrition, and parasite infection

### init 

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import os
from datetime import datetime
import re

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore')

# Custom Color Palette for Microbiome Analysis
# Define a cohesive color palette for consistent visualizations
MICROBIOME_COLORS = {
    'primary': '#2E86AB',        # Blue - for main data
    'secondary': '#A23B72',      # Purple - for secondary data
    'accent': '#F18F01',         # Orange - for highlights
    'success': '#C73E1D',        # Red - for significant results
    'neutral': '#8E9AAF',        # Gray - for neutral elements
    'background': '#F4F4F9',     # Light gray - for backgrounds
    
    # Specific colors for biological categories
    'bacteria': '#2E86AB',       # Blue
    'archaea': '#A23B72',        # Purple
    'fungi': '#F18F01',          # Orange
    'viruses': '#C73E1D',        # Red
    
    # Colors for health status
    'healthy': '#4CAF50',        # Green
    'undernourished': '#FF9800', # Orange
    'infected': '#F44336',       # Red
    'normal': '#2196F3',         # Blue
    
    # Colors for parasites
    'BH': '#E91E63',            # Pink - Blastocystis hominis
    'GL': '#9C27B0',            # Purple - Giardia lamblia  
    'DF': '#673AB7',            # Deep Purple - Dientamoeba fragilis
    'no_infection': '#4CAF50'   # Green - No infection
}

# Create a qualitative palette for categorical data
CATEGORICAL_PALETTE = [MICROBIOME_COLORS['primary'], MICROBIOME_COLORS['secondary'], 
                      MICROBIOME_COLORS['accent'], MICROBIOME_COLORS['success'], 
                      MICROBIOME_COLORS['healthy'], MICROBIOME_COLORS['BH'], 
                      MICROBIOME_COLORS['GL'], MICROBIOME_COLORS['DF']]

# Set plotting style with custom colors
plt.style.use('default')
sns.set_palette(CATEGORICAL_PALETTE)

print("Libraries imported successfully!")
print(f"Analysis started on: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("Custom microbiome color palette loaded!")

### Checking data is there!

In [ ]:
# Configuration and Setup
# Define custom output directory
custom_outputdir = Path("./microbiome_analysis_output")
custom_outputdir.mkdir(exist_ok=True)

# Create subdirectories for organized output
(custom_outputdir / "data_summaries").mkdir(exist_ok=True)
(custom_outputdir / "quality_control").mkdir(exist_ok=True)
(custom_outputdir / "plots").mkdir(exist_ok=True)
(custom_outputdir / "tables").mkdir(exist_ok=True)

# Define data file paths
data_dir = Path(".")
file_paths = {
    'asv_table': data_dir / "ASV_table.tsv",
    'asv_taxonomy': data_dir / "ASV_tax_species.user.tsv",
    'ec_function': data_dir / "EC_pred_metagenome_unstrat_descrip 1.tsv",
    'ko_function': data_dir / "KO_pred_metagenome_unstrat_descrip 1.tsv",
    'metacyc_pathway': data_dir / "METACYC_path_abun_unstrat_descrip 1.tsv",
    'metadata': data_dir / "data_formicrobiome02042025ok.xlsx"
}

# Verify all files exist
missing_files = []
for name, path in file_paths.items():
    if not path.exists():
        missing_files.append(f"{name}: {path}")
    else:
        print(f"✓ Found {name}: {path}")

if missing_files:
    print("\n❌ Missing files:")
    for missing in missing_files:
        print(f"  - {missing}")
else:
    print(f"\n✅ All files found! Output will be saved to: {custom_outputdir.absolute()}")

### Loading Data

In [ ]:
# Data Loading Functions
def load_and_inspect_data():
    """Load all data files and perform initial inspection"""
    
    print("📁 Loading data files...")
    data = {}
    
    # 1. Load ASV abundance table
    print("  Loading ASV abundance table...")
    data['asv_table'] = pd.read_csv(file_paths['asv_table'], sep='\t', index_col=0)
    print(f"    Shape: {data['asv_table'].shape}")
    
    # 2. Load ASV taxonomy
    print("  Loading ASV taxonomy...")
    data['asv_taxonomy'] = pd.read_csv(file_paths['asv_taxonomy'], sep='\t')
    print(f"    Shape: {data['asv_taxonomy'].shape}")
    
    # 3. Load EC function predictions
    print("  Loading EC function predictions...")
    data['ec_function'] = pd.read_csv(file_paths['ec_function'], sep='\t')
    print(f"    Shape: {data['ec_function'].shape}")
    
    # 4. Load KO function predictions
    print("  Loading KO function predictions...")
    data['ko_function'] = pd.read_csv(file_paths['ko_function'], sep='\t')
    print(f"    Shape: {data['ko_function'].shape}")
    
    # 5. Load MetaCyc pathway predictions
    print("  Loading MetaCyc pathway predictions...")
    data['metacyc_pathway'] = pd.read_csv(file_paths['metacyc_pathway'], sep='\t')
    print(f"    Shape: {data['metacyc_pathway'].shape}")
    
    # 6. Load metadata
    print("  Loading metadata...")
    data['metadata'] = pd.read_excel(file_paths['metadata'])
    print(f"    Shape: {data['metadata'].shape}")
    
    print("✅ All data files loaded successfully!\n")
    return data

# Load the data
raw_data = load_and_inspect_data()

### Evaluating data nad standarizing

In [ ]:
# Sample ID Standardization Functions
def standardize_sample_ids():
    """Standardize sample IDs across all datasets and identify mismatches"""
    
    print("🔧 Standardizing sample IDs...")
    
    # Extract sample IDs from each dataset
    sample_ids = {}
    
    # ASV table - columns (except first which is ASV_ID)
    sample_ids['asv_table'] = list(raw_data['asv_table'].columns)
    print(f"  ASV table samples: {len(sample_ids['asv_table'])}")
    
    # EC function - columns (except 'function' and 'description')
    ec_cols = [col for col in raw_data['ec_function'].columns if col not in ['function', 'description']]
    sample_ids['ec_function'] = ec_cols
    print(f"  EC function samples: {len(sample_ids['ec_function'])}")
    
    # KO function - columns (except 'function' and 'description')
    ko_cols = [col for col in raw_data['ko_function'].columns if col not in ['function', 'description']]
    sample_ids['ko_function'] = ko_cols
    print(f"  KO function samples: {len(sample_ids['ko_function'])}")
    
    # MetaCyc pathway - columns (except 'pathway' and 'description')
    metacyc_cols = [col for col in raw_data['metacyc_pathway'].columns if col not in ['pathway', 'description']]
    sample_ids['metacyc_pathway'] = metacyc_cols
    print(f"  MetaCyc pathway samples: {len(sample_ids['metacyc_pathway'])}")
    
    # Metadata - PC_code column
    sample_ids['metadata'] = raw_data['metadata']['PC_code'].astype(str).tolist()
    print(f"  Metadata samples: {len(sample_ids['metadata'])}")
    
    return sample_ids

def find_sample_mismatches(sample_ids):
    """Identify sample ID mismatches between datasets"""
    
    print("\n🔍 Identifying sample ID mismatches...")
    
    # Get all unique sample IDs
    all_samples = set()
    for dataset, samples in sample_ids.items():
        all_samples.update(samples)
    
    print(f"  Total unique sample IDs across all datasets: {len(all_samples)}")
    
    # Find samples present in each dataset
    sample_presence = {}
    for sample in sorted(all_samples):
        sample_presence[sample] = {}
        for dataset, samples in sample_ids.items():
            sample_presence[sample][dataset] = sample in samples
    
    # Create a comprehensive mismatch report
    mismatch_report = []
    complete_samples = []
    
    for sample, presence in sample_presence.items():
        datasets_with_sample = [dataset for dataset, present in presence.items() if present]
        datasets_missing_sample = [dataset for dataset, present in presence.items() if not present]
        
        if len(datasets_with_sample) == len(sample_ids):
            complete_samples.append(sample)
        else:
            mismatch_report.append({
                'sample_id': sample,
                'present_in': datasets_with_sample,
                'missing_from': datasets_missing_sample,
                'n_datasets_present': len(datasets_with_sample),
                'n_datasets_missing': len(datasets_missing_sample)
            })
    
    print(f"  Samples present in ALL datasets: {len(complete_samples)}")
    print(f"  Samples with mismatches: {len(mismatch_report)}")
    
    return sample_presence, mismatch_report, complete_samples

# Perform sample ID standardization
sample_ids = standardize_sample_ids()
sample_presence, mismatch_report, complete_samples = find_sample_mismatches(sample_ids)

### Reporting sample mismatches

In [ ]:
# Sample Mismatch Analysis and Reporting
def create_mismatch_reports():
    """Create detailed reports of sample ID mismatches"""
    
    print("📊 Creating mismatch reports...")
    
    # Save mismatch report as CSV
    if mismatch_report:
        mismatch_df = pd.DataFrame(mismatch_report)
        mismatch_file = custom_outputdir / "quality_control" / "sample_id_mismatches.csv"
        mismatch_df.to_csv(mismatch_file, index=False)
        print(f"  Saved mismatch report to: {mismatch_file}")
        
        # Display top mismatches
        print("\n  Top 10 samples with mismatches:")
        print(mismatch_df.head(10)[['sample_id', 'n_datasets_present', 'missing_from']])
    
    # Save complete samples list
    complete_samples_file = custom_outputdir / "quality_control" / "complete_samples.txt"
    with open(complete_samples_file, 'w') as f:
        for sample in sorted(complete_samples):
            f.write(f"{sample}\n")
    print(f"  Saved complete samples list to: {complete_samples_file}")
    
    # Create sample presence matrix
    presence_matrix = pd.DataFrame(sample_presence).T
    presence_file = custom_outputdir / "quality_control" / "sample_presence_matrix.csv"
    presence_matrix.to_csv(presence_file)
    print(f"  Saved sample presence matrix to: {presence_file}")
    
    # Summary statistics
    print(f"\n📈 Sample ID Summary:")
    print(f"  Total unique samples: {len(sample_presence)}")
    print(f"  Complete samples (all datasets): {len(complete_samples)}")
    print(f"  Samples with mismatches: {len(mismatch_report)}")
    
    for dataset, samples in sample_ids.items():
        print(f"  {dataset}: {len(samples)} samples")
    
    return presence_matrix

# Create and save mismatch reports
presence_matrix = create_mismatch_reports()

### datasets summary

In [ ]:
# Data Summary and Quality Control Functions
def create_data_summaries():
    """Create comprehensive summaries for each dataset"""
    
    print("📋 Creating data summaries...")
    
    summaries = {}
    
    # 1. ASV Table Summary
    print("  Analyzing ASV abundance table...")
    asv_summary = {
        'n_asvs': raw_data['asv_table'].shape[0],
        'n_samples': raw_data['asv_table'].shape[1],
        'total_reads': raw_data['asv_table'].sum().sum(),
        'mean_reads_per_sample': raw_data['asv_table'].sum(axis=0).mean(),
        'median_reads_per_sample': raw_data['asv_table'].sum(axis=0).median(),
        'min_reads_per_sample': raw_data['asv_table'].sum(axis=0).min(),
        'max_reads_per_sample': raw_data['asv_table'].sum(axis=0).max(),
        'zero_abundance_percentage': (raw_data['asv_table'] == 0).sum().sum() / (raw_data['asv_table'].shape[0] * raw_data['asv_table'].shape[1]) * 100
    }
    summaries['asv_table'] = asv_summary
    
    # 2. ASV Taxonomy Summary
    print("  Analyzing ASV taxonomy...")
    tax_summary = {
        'n_asvs_with_taxonomy': raw_data['asv_taxonomy'].shape[0],
        'unique_kingdoms': raw_data['asv_taxonomy']['Kingdom'].nunique(),
        'unique_phyla': raw_data['asv_taxonomy']['Phylum'].nunique(),
        'unique_classes': raw_data['asv_taxonomy']['Class'].nunique(),
        'unique_orders': raw_data['asv_taxonomy']['Order'].nunique(),
        'unique_families': raw_data['asv_taxonomy']['Family'].nunique(),
        'unique_genera': raw_data['asv_taxonomy']['Genus'].nunique(),
        'unique_species': raw_data['asv_taxonomy']['Species'].nunique(),
        'mean_confidence': raw_data['asv_taxonomy']['confidence'].mean(),
        'min_confidence': raw_data['asv_taxonomy']['confidence'].min(),
        'max_confidence': raw_data['asv_taxonomy']['confidence'].max()
    }
    summaries['asv_taxonomy'] = tax_summary
    
    # 3. EC Function Summary
    print("  Analyzing EC functions...")
    ec_cols = [col for col in raw_data['ec_function'].columns if col not in ['function', 'description']]
    ec_data = raw_data['ec_function'][ec_cols]
    ec_summary = {
        'n_ec_functions': raw_data['ec_function'].shape[0],
        'n_samples': len(ec_cols),
        'mean_abundance_per_function': ec_data.mean(axis=1).mean(),
        'median_abundance_per_function': ec_data.mean(axis=1).median(),
        'zero_abundance_percentage': (ec_data == 0).sum().sum() / (ec_data.shape[0] * ec_data.shape[1]) * 100
    }
    summaries['ec_function'] = ec_summary
    
    # 4. KO Function Summary
    print("  Analyzing KO functions...")
    ko_cols = [col for col in raw_data['ko_function'].columns if col not in ['function', 'description']]
    ko_data = raw_data['ko_function'][ko_cols]
    ko_summary = {
        'n_ko_functions': raw_data['ko_function'].shape[0],
        'n_samples': len(ko_cols),
        'mean_abundance_per_function': ko_data.mean(axis=1).mean(),
        'median_abundance_per_function': ko_data.mean(axis=1).median(),
        'zero_abundance_percentage': (ko_data == 0).sum().sum() / (ko_data.shape[0] * ko_data.shape[1]) * 100
    }
    summaries['ko_function'] = ko_summary
    
    # 5. MetaCyc Pathway Summary
    print("  Analyzing MetaCyc pathways...")
    metacyc_cols = [col for col in raw_data['metacyc_pathway'].columns if col not in ['pathway', 'description']]
    metacyc_data = raw_data['metacyc_pathway'][metacyc_cols]
    metacyc_summary = {
        'n_pathways': raw_data['metacyc_pathway'].shape[0],
        'n_samples': len(metacyc_cols),
        'mean_abundance_per_pathway': metacyc_data.mean(axis=1).mean(),
        'median_abundance_per_pathway': metacyc_data.mean(axis=1).median(),
        'zero_abundance_percentage': (metacyc_data == 0).sum().sum() / (metacyc_data.shape[0] * metacyc_data.shape[1]) * 100
    }
    summaries['metacyc_pathway'] = metacyc_summary
    
    # 6. Metadata Summary - EXPANDED to include all relevant columns
    print("  Analyzing metadata...")
    
    # Helper function to safely get value counts
    def safe_value_counts(series):
        if series.notna().sum() > 0:
            return series.value_counts(dropna=False).to_dict()
        else:
            return {"No data": 0}
    
    # Helper function to safely get range for numeric columns
    def safe_range(series):
        if series.notna().sum() > 0:
            return f"{series.min():.1f} - {series.max():.1f}"
        else:
            return "No data"
    
    metadata_summary = {
        'n_samples': raw_data['metadata'].shape[0],
        'n_variables': raw_data['metadata'].shape[1],
        'available_columns': list(raw_data['metadata'].columns),
        'missing_values_per_column': raw_data['metadata'].isnull().sum().to_dict()
    }
    
    # Add summaries for specific columns if they exist
    if 'AgeM' in raw_data['metadata'].columns:
        metadata_summary['age_range_months'] = safe_range(raw_data['metadata']['AgeM'])
    
    if 'Sexchildren' in raw_data['metadata'].columns:
        metadata_summary['sex_distribution'] = safe_value_counts(raw_data['metadata']['Sexchildren'])
    
    if 'Nutritionalstatus' in raw_data['metadata'].columns:
        metadata_summary['nutrition_status_distribution'] = safe_value_counts(raw_data['metadata']['Nutritionalstatus'])
    
    if 'infectioncoinfection' in raw_data['metadata'].columns:
        metadata_summary['infection_distribution'] = safe_value_counts(raw_data['metadata']['infectioncoinfection'])
    
    if 'HEI' in raw_data['metadata'].columns:
        metadata_summary['hei_score_range'] = safe_range(raw_data['metadata']['HEI'])
    
    # ADD MISSING COLUMNS: BH, GL, DF, Dietary_pattern
    if 'BH' in raw_data['metadata'].columns:
        metadata_summary['blastocystis_hominis_distribution'] = safe_value_counts(raw_data['metadata']['BH'])
    
    if 'GL' in raw_data['metadata'].columns:
        metadata_summary['giardia_lamblia_distribution'] = safe_value_counts(raw_data['metadata']['GL'])
    
    if 'DF' in raw_data['metadata'].columns:
        metadata_summary['dientamoeba_fragilis_distribution'] = safe_value_counts(raw_data['metadata']['DF'])
    
    if 'Dietary_pattern' in raw_data['metadata'].columns:
        metadata_summary['dietary_pattern_distribution'] = safe_value_counts(raw_data['metadata']['Dietary_pattern'])
    
    # Add KGcode if available (kindergarten code for blocking)
    if 'KGcode' in raw_data['metadata'].columns:
        metadata_summary['kindergarten_code_distribution'] = safe_value_counts(raw_data['metadata']['KGcode'])
    
    # Add any other potentially relevant columns
    potential_columns = ['BMI', 'Weight', 'Height', 'Ethnicity', 'SES', 'Location']
    for col in potential_columns:
        if col in raw_data['metadata'].columns:
            if raw_data['metadata'][col].dtype in ['int64', 'float64']:
                metadata_summary[f'{col.lower()}_range'] = safe_range(raw_data['metadata'][col])
            else:
                metadata_summary[f'{col.lower()}_distribution'] = safe_value_counts(raw_data['metadata'][col])
    
    summaries['metadata'] = metadata_summary
    
    return summaries

# Create data summaries with the expanded metadata analysis
data_summaries = create_data_summaries()

# Display the enhanced metadata summary
print("\n🔍 ENHANCED METADATA SUMMARY")
print("=" * 50)
for key, value in data_summaries['metadata'].items():
    if key == 'available_columns':
        print(f"{key}: {', '.join(value)}")
    elif isinstance(value, dict) and len(value) > 0:
        print(f"\n{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

### saving dataset sumaries and report

In [ ]:
# Create Data Dictionary and Save Summary Reports
def create_data_dictionary():
    """Create a comprehensive data dictionary"""
    
    print("📚 Creating data dictionary...")
    
    data_dict = {
        'dataset_info': {
            'ASV_table.tsv': {
                'description': 'Amplicon Sequence Variant abundance table',
                'rows': 'ASV identifiers (taxa)',
                'columns': 'Sample identifiers (PC_codes)',
                'values': 'Relative abundance of each ASV in each sample',
                'shape': f"{raw_data['asv_table'].shape[0]} ASVs x {raw_data['asv_table'].shape[1]} samples"
            },
            'ASV_tax_species.user.tsv': {
                'description': 'Taxonomic classification of ASVs',
                'rows': 'ASV identifiers',
                'columns': 'Taxonomic levels + metadata',
                'key_columns': ['ASV_ID', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species', 'confidence'],
                'shape': f"{raw_data['asv_taxonomy'].shape[0]} ASVs x {raw_data['asv_taxonomy'].shape[1]} columns"
            },
            'EC_pred_metagenome_unstrat_descrip 1.tsv': {
                'description': 'Predicted enzyme function abundances (EC codes)',
                'rows': 'EC (Enzyme Commission) identifiers',
                'columns': 'function, description, + sample abundances',
                'values': 'Predicted relative abundance of each enzyme function',
                'shape': f"{raw_data['ec_function'].shape[0]} EC functions x {raw_data['ec_function'].shape[1]} columns"
            },
            'KO_pred_metagenome_unstrat_descrip 1.tsv': {
                'description': 'Predicted KEGG Orthology pathway abundances',
                'rows': 'KO (KEGG Orthology) identifiers',
                'columns': 'function, description, + sample abundances',
                'values': 'Predicted relative abundance of each KO pathway',
                'shape': f"{raw_data['ko_function'].shape[0]} KO functions x {raw_data['ko_function'].shape[1]} columns"
            },
            'METACYC_path_abun_unstrat_descrip 1.tsv': {
                'description': 'Predicted MetaCyc pathway abundances',
                'rows': 'MetaCyc pathway identifiers',
                'columns': 'pathway, description, + sample abundances',
                'values': 'Predicted relative abundance of each metabolic pathway',
                'shape': f"{raw_data['metacyc_pathway'].shape[0]} pathways x {raw_data['metacyc_pathway'].shape[1]} columns"
            },
            'data_formicrobiome02042025ok.xlsx': {
                'description': 'Sample metadata including nutrition, infection, and dietary information',
                'rows': 'Individual samples/children',
                'key_variables': 'PC_code, demographics, nutrition scores, parasite infections, dietary patterns',
                'shape': f"{raw_data['metadata'].shape[0]} samples x {raw_data['metadata'].shape[1]} variables"
            }
        },
        'metadata_variables': {
            'PC_code': 'Sample identifier code',
            'KGcode': 'Kindergarten grouping code',
            'Sexchildren': 'Gender/sex of children',
            'AgeM': 'Age in months',
            'AgeGroup': 'Categorized age groups',
            'WAZ': 'Weight-for-Age Z-score',
            'HAZ': 'Height-for-Age Z-score', 
            'BAZ': 'BMI-for-Age Z-score',
            'Nutritionalstatus': 'Nutritional status based on Z-scores',
            'infectioncoinfection': 'Parasite infection status (single/multiple/none)',
            'HEI': 'Healthy Eating Index score (continuous variable - PRIMARY for analysis)',
            'HEIcategories': 'Categorized HEI scores (for descriptive plots only)',
            'BH': 'Binary variable for Blastocystis hominis infection',
            'GL': 'Binary variable for Giardia lamblia infection', 
            'DF': 'Binary variable for Dientamoeba fragilis infection',
            'Dietary_pattern': 'Custom dietary pattern classification'
        },
        'parasite_info': {
            'BH': {
                'full_name': 'Blastocystis hominis',
                'type': 'Protist parasite',
                'description': 'Common intestinal parasite affecting gut microbiome',
                'color': MICROBIOME_COLORS['BH']
            },
            'GL': {
                'full_name': 'Giardia lamblia',
                'type': 'Flagellated protozoan',
                'description': 'Intestinal parasite causing giardiasis',
                'color': MICROBIOME_COLORS['GL']
            },
            'DF': {
                'full_name': 'Dientamoeba fragilis',
                'type': 'Amoeba parasite',
                'description': 'Intestinal parasite potentially affecting gut health',
                'color': MICROBIOME_COLORS['DF']
            }
        },
        'analysis_guidelines': {
            'HEI_treatment': 'Treat as continuous variable for all statistical analyses',
            'HEI_categories': 'Use only for descriptive visualizations and summaries',
            'parasite_analysis': 'Analyze individual parasites (BH, GL, DF) and combined infection status',
            'recommended_sample_set': 'complete_samples (present in all datasets)',
            'color_palette': 'Use MICROBIOME_COLORS for consistent visualizations'
        },
        'analysis_info': {
            'complete_samples': len(complete_samples),
            'samples_with_mismatches': len(mismatch_report),
            'total_unique_samples': len(sample_presence),
            'recommended_sample_set': 'complete_samples (present in all datasets)',
            'analysis_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
    }
    
    return data_dict

def save_summary_reports(data_summaries, data_dict):
    """Save all summary reports to files"""
    
    print("💾 Saving summary reports...")
    
    # Save data summaries as JSON
    import json
    summaries_file = custom_outputdir / "data_summaries" / "dataset_summaries.json"
    with open(summaries_file, 'w') as f:
        json.dump(data_summaries, f, indent=2, default=str)
    print(f"  Saved dataset summaries to: {summaries_file}")
    
    # Save data dictionary as JSON
    dict_file = custom_outputdir / "data_summaries" / "data_dictionary.json"
    with open(dict_file, 'w') as f:
        json.dump(data_dict, f, indent=2, default=str)
    print(f"  Saved data dictionary to: {dict_file}")
    
    # Create a formatted summary report
    report_file = custom_outputdir / "data_summaries" / "analysis_summary_report.txt"
    with open(report_file, 'w') as f:
        f.write("MICROBIOME FUNCTIONAL ANALYSIS - SUMMARY REPORT\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("DATASET OVERVIEW:\n")
        f.write("-" * 20 + "\n")
        for dataset, summary in data_summaries.items():
            f.write(f"\n{dataset.upper()}:\n")
            for key, value in summary.items():
                f.write(f"  {key}: {value}\n")
        
        f.write(f"\n\nSAMPLE ID ANALYSIS:\n")
        f.write("-" * 20 + "\n")
        f.write(f"Complete samples (all datasets): {len(complete_samples)}\n")
        f.write(f"Samples with mismatches: {len(mismatch_report)}\n")
        f.write(f"Total unique samples: {len(sample_presence)}\n")
        
        f.write(f"\n\nPARASITE INFORMATION:\n")
        f.write("-" * 20 + "\n")
        f.write(f"BH: Blastocystis hominis\n")
        f.write(f"GL: Giardia lamblia\n")
        f.write(f"DF: Dientamoeba fragilis\n")
        
        f.write(f"\n\nANALYSIS GUIDELINES:\n")
        f.write("-" * 20 + "\n")
        f.write(f"• HEI: Treat as continuous variable (primary analysis)\n")
        f.write(f"• HEI categories: Use only for descriptive plots\n")
        f.write(f"• Parasites: Analyze individually and in combination\n")
        f.write(f"• Colors: Use MICROBIOME_COLORS palette for consistency\n")
        
        if complete_samples:
            f.write(f"\nRECOMMENDED ANALYSIS SET: {len(complete_samples)} samples\n")
            f.write("These samples are present in all datasets and recommended for analysis.\n")
    
    print(f"  Saved formatted report to: {report_file}")

# Create data dictionary and save reports
data_dict = create_data_dictionary()
save_summary_reports(data_summaries, data_dict)

### Quick data visualization

In [ ]:
# Quick Data Exploration and Visualization
def create_summary_visualizations():
    """Create summary visualizations for data overview with custom colors"""
    
    print("📊 Creating summary visualizations...")
    
    # Set up the plotting area
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Microbiome Data Overview', fontsize=16, fontweight='bold', color=MICROBIOME_COLORS['primary'])
    
    # 1. Sample distribution across datasets
    dataset_counts = [len(samples) for samples in sample_ids.values()]
    dataset_names = list(sample_ids.keys())
    
    bars1 = axes[0, 0].bar(range(len(dataset_names)), dataset_counts, color=MICROBIOME_COLORS['primary'])
    axes[0, 0].set_title('Sample Counts by Dataset', fontweight='bold')
    axes[0, 0].set_xlabel('Dataset')
    axes[0, 0].set_ylabel('Number of Samples')
    axes[0, 0].set_xticks(range(len(dataset_names)))
    axes[0, 0].set_xticklabels(dataset_names, rotation=45, ha='right')
    
    # Add value labels on bars
    for bar in bars1:
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 1,
                       f'{int(height)}', ha='center', va='bottom')
    
    # 2. Taxonomic diversity (top phyla)
    if 'Phylum' in raw_data['asv_taxonomy'].columns:
        phyla_counts = raw_data['asv_taxonomy']['Phylum'].value_counts().head(10)
        colors_phyla = plt.cm.Set3(np.linspace(0, 1, len(phyla_counts)))
        bars2 = axes[0, 1].barh(range(len(phyla_counts)), phyla_counts.values, color=colors_phyla)
        axes[0, 1].set_title('Top 10 Phyla (ASV count)', fontweight='bold')
        axes[0, 1].set_xlabel('Number of ASVs')
        axes[0, 1].set_yticks(range(len(phyla_counts)))
        axes[0, 1].set_yticklabels(phyla_counts.index)
        axes[0, 1].invert_yaxis()
    
    # 3. Age distribution
    if 'AgeM' in raw_data['metadata'].columns:
        ages = raw_data['metadata']['AgeM'].dropna()
        axes[0, 2].hist(ages, bins=20, color=MICROBIOME_COLORS['accent'], alpha=0.7, edgecolor='black')
        axes[0, 2].set_title('Age Distribution (months)', fontweight='bold')
        axes[0, 2].set_xlabel('Age (months)')
        axes[0, 2].set_ylabel('Number of Children')
        axes[0, 2].axvline(ages.mean(), color=MICROBIOME_COLORS['primary'], linestyle='--', linewidth=2, label=f'Mean: {ages.mean():.1f}')
        axes[0, 2].legend()
    
    # 4. Nutritional status distribution
    if 'Nutritionalstatus' in raw_data['metadata'].columns:
        nutrition_counts = raw_data['metadata']['Nutritionalstatus'].value_counts()
        colors_nutrition = [MICROBIOME_COLORS['healthy'], MICROBIOME_COLORS['undernourished'], MICROBIOME_COLORS['infected']][:len(nutrition_counts)]
        wedges, texts, autotexts = axes[1, 0].pie(nutrition_counts.values, labels=nutrition_counts.index, 
                                                 autopct='%1.1f%%', colors=colors_nutrition)
        axes[1, 0].set_title('Nutritional Status Distribution', fontweight='bold')
        # Make percentage text bold
        for autotext in autotexts:
            autotext.set_fontweight('bold')
            autotext.set_color('white')
    
    # 5. Parasite infection distribution (individual parasites)
    parasite_data = []
    parasite_colors = []
    parasite_labels = []
    
    if all(col in raw_data['metadata'].columns for col in ['BH', 'GL', 'DF']):
        for parasite, full_name in [('BH', 'Blastocystis hominis'), ('GL', 'Giardia lamblia'), ('DF', 'Dientamoeba fragilis')]:
            if parasite in raw_data['metadata'].columns:
                infected_count = raw_data['metadata'][parasite].sum()
                parasite_data.append(infected_count)
                parasite_colors.append(MICROBIOME_COLORS[parasite])
                parasite_labels.append(f'{parasite}\n({full_name})')
        
        bars5 = axes[1, 1].bar(range(len(parasite_data)), parasite_data, color=parasite_colors)
        axes[1, 1].set_title('Individual Parasite Infections', fontweight='bold')
        axes[1, 1].set_xlabel('Parasite Type')
        axes[1, 1].set_ylabel('Number of Infected Children')
        axes[1, 1].set_xticks(range(len(parasite_labels)))
        axes[1, 1].set_xticklabels(parasite_labels, rotation=45, ha='right')
        
        # Add value labels on bars
        for bar in bars5:
            height = bar.get_height()
            axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                           f'{int(height)}', ha='center', va='bottom', fontweight='bold')
    
    # 6. HEI score distribution (continuous)
    if 'HEI' in raw_data['metadata'].columns:
        hei_scores = raw_data['metadata']['HEI'].dropna()
        n, bins, patches = axes[1, 2].hist(hei_scores, bins=15, color=MICROBIOME_COLORS['secondary'], 
                                          alpha=0.7, edgecolor='black')
        axes[1, 2].set_title('Healthy Eating Index Distribution (Continuous)', fontweight='bold')
        axes[1, 2].set_xlabel('HEI Score')
        axes[1, 2].set_ylabel('Number of Children')
        
        # Add statistics
        mean_hei = hei_scores.mean()
        median_hei = hei_scores.median()
        axes[1, 2].axvline(mean_hei, color=MICROBIOME_COLORS['primary'], linestyle='--', linewidth=2, label=f'Mean: {mean_hei:.1f}')
        axes[1, 2].axvline(median_hei, color=MICROBIOME_COLORS['success'], linestyle='-', linewidth=2, label=f'Median: {median_hei:.1f}')
        axes[1, 2].legend()
        
        # Color bars based on HEI quality (if categories exist)
        if 'HEIcategories' in raw_data['metadata'].columns:
            # This is for visual reference only - analysis should use continuous HEI
            hei_text = f"Note: Analysis uses continuous HEI\n(Categories for visualization only)"
            axes[1, 2].text(0.02, 0.98, hei_text, transform=axes[1, 2].transAxes, 
                           verticalalignment='top', fontsize=8, style='italic',
                           bbox=dict(boxstyle='round', facecolor=MICROBIOME_COLORS['background'], alpha=0.8))
    
    plt.tight_layout()
    
    # Save the plot
    plot_file = custom_outputdir / "plots" / "data_overview.png"
    plt.savefig(plot_file, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"  Saved overview plot to: {plot_file}")
    plt.show()

def create_parasite_correlation_matrix():
    """Create a correlation matrix for parasite infections"""
    
    print("🦠 Creating parasite correlation analysis...")
    
    if all(col in raw_data['metadata'].columns for col in ['BH', 'GL', 'DF']):
        # Create correlation matrix for parasites
        parasite_cols = ['BH', 'GL', 'DF']
        parasite_data = raw_data['metadata'][parasite_cols]
        
        # Calculate correlation matrix
        corr_matrix = parasite_data.corr()
        
        # Create visualization
        plt.figure(figsize=(8, 6))
        mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
        
        sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdYlBu_r', center=0,
                   square=True, linewidths=0.5, cbar_kws={"shrink": .8},
                   xticklabels=['Blastocystis hominis', 'Giardia lamblia', 'Dientamoeba fragilis'],
                   yticklabels=['Blastocystis hominis', 'Giardia lamblia', 'Dientamoeba fragilis'])
        
        plt.title('Parasite Co-infection Correlation Matrix', fontweight='bold', fontsize=14)
        plt.tight_layout()
        
        # Save the plot
        corr_file = custom_outputdir / "plots" / "parasite_correlation_matrix.png"
        plt.savefig(corr_file, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"  Saved parasite correlation matrix to: {corr_file}")
        plt.show()
        
        # Print co-infection statistics
        print("\n🔬 Parasite Co-infection Statistics:")
        total_samples = len(raw_data['metadata'])
        
        for parasite, full_name in [('BH', 'Blastocystis hominis'), ('GL', 'Giardia lamblia'), ('DF', 'Dientamoeba fragilis')]:
            infected = raw_data['metadata'][parasite].sum()
            percentage = (infected / total_samples) * 100
            print(f"  {parasite} ({full_name}): {infected}/{total_samples} ({percentage:.1f}%)")
        
        # Co-infection analysis
        multiple_infections = (raw_data['metadata'][parasite_cols].sum(axis=1) > 1).sum()
        no_infections = (raw_data['metadata'][parasite_cols].sum(axis=1) == 0).sum()
        
        print(f"\n  Co-infections (multiple parasites): {multiple_infections} ({(multiple_infections/total_samples)*100:.1f}%)")
        print(f"  No parasite infections: {no_infections} ({(no_infections/total_samples)*100:.1f}%)")

def display_final_summary():
    """Display final summary of the analysis with updated information"""
    
    print("\n" + "="*80)
    print("🎉 DATA LOADING AND STANDARDIZATION COMPLETE!")
    print("="*80)
    
    print(f"\n📁 OUTPUT DIRECTORY: {custom_outputdir.absolute()}")
    print("\n📋 GENERATED FILES:")
    print("   Quality Control:")
    print("   ├── sample_id_mismatches.csv")
    print("   ├── complete_samples.txt") 
    print("   └── sample_presence_matrix.csv")
    print("\n   Data Summaries:")
    print("   ├── dataset_summaries.json")
    print("   ├── data_dictionary.json")
    print("   └── analysis_summary_report.txt")
    print("\n   Visualizations:")
    print("   ├── data_overview.png")
    print("   └── parasite_correlation_matrix.png")
    
    print(f"\n📊 KEY STATISTICS:")
    print(f"   • Total datasets: 6")
    print(f"   • Complete samples (all datasets): {len(complete_samples)}")
    print(f"   • Total unique ASVs: {raw_data['asv_table'].shape[0]}")
    print(f"   • Total EC functions: {raw_data['ec_function'].shape[0]}")
    print(f"   • Total KO functions: {raw_data['ko_function'].shape[0]}")
    print(f"   • Total MetaCyc pathways: {raw_data['metacyc_pathway'].shape[0]}")
    
    print(f"\n🦠 PARASITE INFORMATION:")
    print(f"   • BH: Blastocystis hominis")
    print(f"   • GL: Giardia lamblia")
    print(f"   • DF: Dientamoeba fragilis")
    
    print(f"\n🎨 ANALYSIS SPECIFICATIONS:")
    print(f"   • HEI: Treat as CONTINUOUS variable (primary analysis)")
    print(f"   • HEI categories: Use only for descriptive visualizations")
    print(f"   • Custom color palette: MICROBIOME_COLORS applied")
    print(f"   • Parasite analysis: Individual and combined infection patterns")
    
    print(f"\n🔬 RECOMMENDED NEXT STEPS:")
    print("   1. Use 'complete_samples' list for downstream analysis")
    print("   2. Filter datasets to include only complete samples")
    print("   3. Analyze HEI as continuous variable in all statistical tests")
    print("   4. Explore correlations between microbiome functions and:")
    print("      • Nutritional status (WAZ, HAZ, BAZ)")
    print("      • Individual parasite infections (BH, GL, DF)")
    print("      • Continuous HEI scores (primary)")
    print("      • Dietary patterns")
    print("   5. Consider data normalization and transformation")
    print("   6. Perform statistical testing with proper multiple correction")
    print("   7. Use custom color palette for consistent visualizations")

# Create visualizations and display summary
create_summary_visualizations()
create_parasite_correlation_matrix()
display_final_summary()

### Quality control

In [ ]:
asv_samples = [s for s in samples_to_use if s in raw_data['asv_table'].columns]
clean_data['asv_table'] = raw_data['asv_table'][asv_samples].copy()
# Add ASV_ID as a column from the index
clean_data['asv_table'].insert(0, 'ASV_ID', clean_data['asv_table'].index)
clean_data['asv_table_transpose'] = clean_data['asv_table'].T
clean_data['asv_table'].insert(1, 'description', '')

print(clean_data['asv_table'].iloc[0:3, 0:3])
print(clean_data['asv_table_transpose'].iloc[0:3, 0:3])
print(clean_data['asv_table_transpose'].index)

    


In [ ]:
# Helper Functions for Analysis
def get_clean_dataset(complete_samples_only=True):
    """
    Get cleaned and filtered datasets ready for analysis
    
    Parameters:
    -----------
    complete_samples_only : bool
        If True, filter to only include samples present in all datasets
    
    Returns:
    --------
    dict : Dictionary containing cleaned datasets
    """
    
    print("🧹 Preparing clean datasets for analysis...")
    
    clean_data = {}
    
    # Determine which samples to use
    if complete_samples_only:
        samples_to_use = complete_samples
        print(f"  Using {len(samples_to_use)} complete samples (present in all datasets)")
    else:
        samples_to_use = list(set().union(*[samples for samples in sample_ids.values()]))
        print(f"  Using {len(samples_to_use)} total unique samples")
    
    # Filter ASV table
    asv_samples = [s for s in samples_to_use if s in raw_data['asv_table'].columns]
    clean_data['asv_table'] = raw_data['asv_table'][asv_samples].copy()
    # Add ASV_ID as a column from the index
    clean_data['asv_table'].insert(0, 'ASV_ID', clean_data['asv_table'].index)
    clean_data['asv_table_transpose'] = clean_data['asv_table'].T
    clean_data['asv_table'].insert(1, 'description', '')
    print(f"  ASV table: {clean_data['asv_table'].shape[0]} ASVs x {len(asv_samples)} samples")
    
    # Filter functional datasets
    for func_type in ['ec_function', 'ko_function']:
        func_cols = [col for col in raw_data[func_type].columns if col not in ['function', 'description']]
        available_samples = [s for s in samples_to_use if s in func_cols]
        clean_data[func_type] = raw_data[func_type][['function', 'description'] + available_samples]
        print(f"  {func_type}: {clean_data[func_type].shape[0]} functions x {len(available_samples)} samples")
    
    # Filter MetaCyc pathway data
    metacyc_cols = [col for col in raw_data['metacyc_pathway'].columns if col not in ['pathway', 'description']]
    available_samples = [s for s in samples_to_use if s in metacyc_cols]
    clean_data['metacyc_pathway'] = raw_data['metacyc_pathway'][['pathway', 'description'] + available_samples]
    print(f"  MetaCyc pathways: {clean_data['metacyc_pathway'].shape[0]} pathways x {len(available_samples)} samples")
    
    # Filter metadata
    metadata_samples = raw_data['metadata'][raw_data['metadata']['PC_code'].astype(str).isin(samples_to_use)]
    clean_data['metadata'] = metadata_samples.copy()
    print(f"  Metadata: {clean_data['metadata'].shape[0]} samples x {clean_data['metadata'].shape[1]} variables")
    
    # Add ASV taxonomy (no filtering needed as it's reference data)
    clean_data['asv_taxonomy'] = raw_data['asv_taxonomy'].copy()
    
    return clean_data

def create_parasite_infection_summary(metadata_df):
    """
    Create comprehensive parasite infection summary
    
    Parameters:
    -----------
    metadata_df : DataFrame
        Metadata containing parasite infection data
    
    Returns:
    --------
    dict : Summary of parasite infections
    """
    
    parasites = {
        'BH': 'Blastocystis hominis',
        'GL': 'Giardia lamblia', 
        'DF': 'Dientamoeba fragilis'
    }
    
    summary = {
        'individual_infections': {},
        'co_infections': {},
        'infection_patterns': {}
    }
    
    total_samples = len(metadata_df)
    
    # Individual parasite prevalence
    for code, full_name in parasites.items():
        if code in metadata_df.columns:
            infected = metadata_df[code].sum()
            prevalence = (infected / total_samples) * 100
            summary['individual_infections'][code] = {
                'full_name': full_name,
                'infected_count': int(infected),
                'prevalence_percent': round(prevalence, 1),
                'color': MICROBIOME_COLORS[code]
            }
    
    # Co-infection analysis
    parasite_cols = [p for p in parasites.keys() if p in metadata_df.columns]
    if parasite_cols:
        infection_counts = metadata_df[parasite_cols].sum(axis=1)
        
        summary['infection_patterns'] = {
            'no_infection': int((infection_counts == 0).sum()),
            'single_infection': int((infection_counts == 1).sum()),
            'dual_infection': int((infection_counts == 2).sum()),
            'triple_infection': int((infection_counts == 3).sum())
        }
        
        # Specific co-infection combinations
        for i, p1 in enumerate(parasite_cols):
            for j, p2 in enumerate(parasite_cols[i+1:], i+1):
                co_infected = ((metadata_df[p1] == 1) & (metadata_df[p2] == 1)).sum()
                summary['co_infections'][f'{p1}_{p2}'] = {
                    'parasites': f'{parasites[p1]} + {parasites[p2]}',
                    'count': int(co_infected),
                    'prevalence_percent': round((co_infected / total_samples) * 100, 1)
                }
    
    return summary

def prepare_hei_analysis_data(metadata_df):
    """
    Prepare HEI data for analysis (continuous primary, categorical for description)
    
    Parameters:
    -----------
    metadata_df : DataFrame
        Metadata containing HEI information
    
    Returns:
    --------
    dict : HEI analysis data
    """
    
    hei_data = {
        'continuous': None,
        'categories': None,
        'statistics': {},
        'recommendations': []
    }
    
    if 'HEI' in metadata_df.columns:
        # Continuous HEI (primary for analysis)
        hei_continuous = metadata_df['HEI'].dropna()
        hei_data['continuous'] = hei_continuous
        
        # Calculate statistics
        hei_data['statistics'] = {
            'count': len(hei_continuous),
            'mean': round(hei_continuous.mean(), 2),
            'median': round(hei_continuous.median(), 2),
            'std': round(hei_continuous.std(), 2),
            'min': round(hei_continuous.min(), 2),
            'max': round(hei_continuous.max(), 2),
            'q25': round(hei_continuous.quantile(0.25), 2),
            'q75': round(hei_continuous.quantile(0.75), 2)
        }
        
        # Categorical HEI (for descriptive purposes only)
        if 'HEIcategories' in metadata_df.columns:
            hei_data['categories'] = metadata_df['HEIcategories'].value_counts()
        
        # Analysis recommendations
        hei_data['recommendations'] = [
            "Use continuous HEI scores for all statistical analyses",
            "Treat HEI as a continuous predictor in regression models",
            "Use HEI categories only for descriptive visualizations",
            "Consider HEI score transformations if distribution is skewed",
            "Examine HEI relationships with microbiome diversity metrics"
        ]
    
    return hei_data

def save_analysis_ready_data(clean_data, output_dir):
    """
    Save cleaned datasets in analysis-ready format
    
    Parameters:
    -----------
    clean_data : dict
        Dictionary of cleaned datasets
    output_dir : Path
        Output directory for saving files
    """
    
    print("💾 Saving analysis-ready datasets...")
    
    # Create analysis data directory
    analysis_dir = output_dir / "analysis_ready_data"
    analysis_dir.mkdir(exist_ok=True)
    
    # Save each dataset
    for dataset_name, dataset in clean_data.items():
        if dataset_name == 'asv_table_transpose':
            # Transpose ASV table for easier analysis (samples as rows)
            file_path = analysis_dir / f"{dataset_name}_clean.csv"
            {dataset_name}.to_csv(file_path, index=True)
        else:
            file_path = analysis_dir / f"{dataset_name}_clean.csv"
            dataset.to_csv(file_path, index=False)
        
        print(f"  Saved {dataset_name}: {file_path}")
    
    print(f"\n✅ All analysis-ready datasets saved to: {analysis_dir}")

print("Helper functions loaded successfully!")
print("\nAvailable functions:")
print("- get_clean_dataset(): Get filtered datasets ready for analysis")
print("- create_parasite_infection_summary(): Analyze parasite infection patterns") 
print("- prepare_hei_analysis_data(): Prepare HEI data for continuous analysis")
print("- save_analysis_ready_data(): Save cleaned datasets for analysis")

### Data input and summary exec!

In [ ]:
# Execute Analysis with Updated Specifications
print("🚀 Executing comprehensive analysis with updated specifications...")
print("=" * 70)

# Get clean datasets
clean_data = get_clean_dataset(complete_samples_only=True)

# Analyze parasite infections with proper names
parasite_summary = create_parasite_infection_summary(clean_data['metadata'])

print("\n🦠 PARASITE INFECTION ANALYSIS:")
print("-" * 35)
for code, info in parasite_summary['individual_infections'].items():
    print(f"  {code} ({info['full_name']}):")
    print(f"    Infected: {info['infected_count']} children ({info['prevalence_percent']}%)")

print("\n  Co-infection patterns:")
for pattern, count in parasite_summary['infection_patterns'].items():
    print(f"    {pattern.replace('_', ' ').title()}: {count} children")

# Analyze HEI as continuous variable
hei_analysis = prepare_hei_analysis_data(clean_data['metadata'])

print("\n📊 HEI ANALYSIS (Continuous Primary):")
print("-" * 35)
if hei_analysis['continuous'] is not None:
    stats = hei_analysis['statistics']
    print(f"  Samples with HEI data: {stats['count']}")
    print(f"  Mean HEI score: {stats['mean']} ± {stats['std']}")
    print(f"  Median HEI score: {stats['median']}")
    print(f"  Range: {stats['min']} - {stats['max']}")
    print(f"  IQR: {stats['q25']} - {stats['q75']}")
    
    print("\n  Analysis recommendations:")
    for rec in hei_analysis['recommendations']:
        print(f"    • {rec}")

# Save analysis-ready datasets
save_analysis_ready_data(clean_data, custom_outputdir)

print("\n🎨 COLOR PALETTE INFORMATION:")
print("-" * 30)
print("  Custom MICROBIOME_COLORS loaded with:")
print(f"    • BH (Blastocystis hominis): {MICROBIOME_COLORS['BH']}")
print(f"    • GL (Giardia lamblia): {MICROBIOME_COLORS['GL']}")  
print(f"    • DF (Dientamoeba fragilis): {MICROBIOME_COLORS['DF']}")
print(f"    • Primary analysis color: {MICROBIOME_COLORS['primary']}")
print(f"    • Healthy status: {MICROBIOME_COLORS['healthy']}")

print("\n" + "=" * 70)
print("✅ ANALYSIS SETUP COMPLETE!")
print("All datasets loaded, standardized, and ready for microbiome analysis")
print("with proper parasite naming, continuous HEI treatment, and custom colors.")
print("=" * 70)

### visualization 

In [ ]:
# Visualization and Execution Functions for Preprocessing Pipeline

def visualize_preprocessing_effects(pipeline_results, dataset_name="Dataset"):
    """
    Create comprehensive visualizations of preprocessing effects
    
    Parameters:
    -----------
    pipeline_results : dict
        Results from apply_full_preprocessing_pipeline()
    dataset_name : str
        Name of the dataset for plot titles
    """
    
    print(f"📊 Creating preprocessing visualizations for {dataset_name}...")
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Preprocessing Effects: {dataset_name}', fontsize=16, y=0.98)
    
    # Colors from our custom palette
    colors = [MICROBIOME_COLORS['primary'], MICROBIOME_COLORS['secondary'], 
              MICROBIOME_COLORS['accent'], MICROBIOME_COLORS['success']]
    
    # 1. Feature count reduction
    steps = ['Original'] + [step.replace('_', ' ').title() for step in pipeline_results['preprocessing_steps']]
    feature_counts = [pipeline_results['pipeline_stats']['original_features']]
    
    if 'prevalence_stats' in pipeline_results:
        feature_counts.append(pipeline_results['prevalence_stats']['filtered_features'])
    if 'variance_stats' in pipeline_results:
        feature_counts.append(pipeline_results['variance_stats']['filtered_features'])
    
    # Pad with final count if needed
    while len(feature_counts) < len(steps):
        feature_counts.append(pipeline_results['pipeline_stats']['final_features'])
    
    axes[0, 0].bar(range(len(steps)), feature_counts, color=colors[:len(steps)])
    axes[0, 0].set_xlabel('Preprocessing Step')
    axes[0, 0].set_ylabel('Number of Features')
    axes[0, 0].set_title('Feature Count Reduction')
    axes[0, 0].set_xticks(range(len(steps)))
    axes[0, 0].set_xticklabels(steps, rotation=45, ha='right')
    
    # Add percentage labels
    for i, count in enumerate(feature_counts):
        pct = (count / feature_counts[0]) * 100
        axes[0, 0].text(i, count + max(feature_counts) * 0.01, f'{pct:.1f}%', 
                       ha='center', va='bottom', fontweight='bold')
    
    # 2. Prevalence distribution (if available)
    if 'prevalence_stats' in pipeline_results:
        prevalence_dist = pipeline_results['prevalence_stats']['prevalence_distribution']
        prev_data = [prevalence_dist['min'], prevalence_dist['q25'], 
                    prevalence_dist['median'], prevalence_dist['q75'], prevalence_dist['max']]
        prev_labels = ['Min', 'Q25', 'Median', 'Q75', 'Max']
        
        axes[0, 1].bar(prev_labels, prev_data, color=MICROBIOME_COLORS['primary'], alpha=0.7)
        axes[0, 1].axhline(y=pipeline_results['prevalence_stats']['min_prevalence_threshold'], 
                          color=MICROBIOME_COLORS['success'], linestyle='--', 
                          label=f"Threshold ({pipeline_results['prevalence_stats']['min_prevalence_threshold']}%)")
        axes[0, 1].set_ylabel('Prevalence (%)')
        axes[0, 1].set_title('Feature Prevalence Distribution')
        axes[0, 1].legend()
    
    # 3. Variance distribution (if available)
    if 'variance_stats' in pipeline_results:
        var_dist = pipeline_results['variance_stats']['variance_distribution']
        var_data = [var_dist['min'], var_dist['q25'], 
                   var_dist['median'], var_dist['q75'], var_dist['max']]
        var_labels = ['Min', 'Q25', 'Median', 'Q75', 'Max']
        
        axes[0, 2].bar(var_labels, var_data, color=MICROBIOME_COLORS['secondary'], alpha=0.7)
        axes[0, 2].axhline(y=pipeline_results['variance_stats']['variance_threshold'], 
                          color=MICROBIOME_COLORS['success'], linestyle='--', 
                          label=f"Threshold ({pipeline_results['variance_stats']['variance_threshold']:.4f})")
        axes[0, 2].set_ylabel('Aitchison Variance')
        axes[0, 2].set_title('Feature Variance Distribution')
        axes[0, 2].legend()
    
    # 4. Pseudocount effects (if available)
    if 'pseudocount_stats' in pipeline_results:
        ps_stats = pipeline_results['pseudocount_stats']
        zero_data = [ps_stats['zeros_before'], ps_stats['zeros_after']]
        zero_labels = ['Before Pseudocount', 'After Pseudocount']
        
        axes[1, 0].bar(zero_labels, zero_data, color=[MICROBIOME_COLORS['accent'], MICROBIOME_COLORS['primary']])
        axes[1, 0].set_ylabel('Number of Zero Values')
        axes[1, 0].set_title(f"Zero Value Handling\\n(Pseudocount: {ps_stats['pseudocount_value']:.2e})")
        
        # Add percentage labels
        for i, count in enumerate(zero_data):
            pct = (count / ps_stats['total_values']) * 100
            axes[1, 0].text(i, count + max(zero_data) * 0.01, f'{pct:.1f}%', 
                           ha='center', va='bottom', fontweight='bold')
    
    # 5. CLR transformation validation (if available)
    if 'clr_stats' in pipeline_results:
        clr_validation = pipeline_results['clr_stats']['clr_validation']
        
        # Create a simple validation summary
        validation_items = ['Zero-sum Property', 'Scale Invariant', 'Numeric Stability']
        validation_scores = [
            1 if clr_validation['zero_sum_check'] else 0,
            1 if clr_validation['scale_invariant'] else 0,
            1 if clr_validation['max_abs_col_sum'] < 1e-6 else 0
        ]
        
        colors_val = [MICROBIOME_COLORS['healthy'] if score > 0.5 else MICROBIOME_COLORS['success'] 
                     for score in validation_scores]
        
        axes[1, 1].bar(validation_items, validation_scores, color=colors_val)
        axes[1, 1].set_ylabel('Validation Score')
        axes[1, 1].set_title('CLR Transformation Validation')
        axes[1, 1].set_ylim(0, 1.2)
        axes[1, 1].set_xticklabels(validation_items, rotation=45, ha='right')
        
        # Add validation text
        axes[1, 1].text(0.5, 0.5, f"Max |Col Sum|: {clr_validation['max_abs_col_sum']:.2e}", 
                        ha='center', va='center', transform=axes[1, 1].transAxes,
                        bbox=dict(boxstyle="round,pad=0.3", facecolor=MICROBIOME_COLORS['background']))
    
    # 6. Overall pipeline summary
    pipeline_stats = pipeline_results['pipeline_stats']
    summary_text = f"""Pipeline Summary:
    
Original Features: {pipeline_stats['original_features']:,}
Final Features: {pipeline_stats['final_features']:,}
Removed: {pipeline_stats['total_features_removed']:,} ({pipeline_stats['total_removal_percentage']:.1f}%)

Steps Applied:
{chr(10).join([f"• {step.replace('_', ' ').title()}" for step in pipeline_stats['preprocessing_steps_applied']])}

Feature Column: {pipeline_stats.get('feature_column_used', 'Auto-detected')}
"""
    
    axes[1, 2].text(0.05, 0.95, summary_text, transform=axes[1, 2].transAxes,
                    verticalalignment='top', fontfamily='monospace', fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.5", facecolor=MICROBIOME_COLORS['background']))
    axes[1, 2].set_xlim(0, 1)
    axes[1, 2].set_ylim(0, 1)
    axes[1, 2].axis('off')
    axes[1, 2].set_title('Pipeline Summary')
    
    plt.tight_layout()
    plt.show()
    
    print(f"✅ Preprocessing visualization completed for {dataset_name}")

def save_preprocessed_data(pipeline_results, dataset_name, output_dir):
    """
    Save preprocessed data at each step for further analysis
    
    Parameters:
    -----------
    pipeline_results : dict
        Results from preprocessing pipeline
    dataset_name : str
        Name of the dataset
    output_dir : str or Path
        Output directory for saved files
    """
    
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    print(f"💾 Saving preprocessed {dataset_name} data...")
    
    # Save data at each preprocessing step
    for step_name, data in pipeline_results.items():
        if isinstance(data, pd.DataFrame) and step_name != 'original_data':
            file_path = output_path / f"{dataset_name}_{step_name}.csv"
            data.to_csv(file_path, index=False)
            print(f"  Saved {step_name}: {file_path}")
    
    # Save pipeline statistics
    stats_file = output_path / f"{dataset_name}_preprocessing_stats.json"
    import json
    
    # Convert numpy types to native Python types for JSON serialization
    def convert_types(obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {key: convert_types(value) for key, value in obj.items()}
        elif isinstance(obj, list):
            return [convert_types(item) for item in obj]
        return obj
    
    # Extract serializable statistics
    serializable_stats = {
        'pipeline_stats': convert_types(pipeline_results['pipeline_stats']),
        'preprocessing_steps': pipeline_results['preprocessing_steps']
    }
    
    # Add individual step statistics if available
    for stat_key in ['prevalence_stats', 'variance_stats', 'pseudocount_stats']:
        if stat_key in pipeline_results:
            serializable_stats[stat_key] = convert_types(pipeline_results[stat_key])
    
    with open(stats_file, 'w') as f:
        json.dump(serializable_stats, f, indent=2)
    
    print(f"  Saved statistics: {stats_file}")
    print(f"✅ All {dataset_name} preprocessing files saved to: {output_path}")

print("Visualization and execution functions loaded successfully!")
print("\\nAvailable functions:")
print("- visualize_preprocessing_effects(): Create comprehensive preprocessing visualizations")
print("- save_preprocessed_data(): Save preprocessed data and statistics")

### Execute Improved Preprocessing Pipeline

Now let's run the improved preprocessing pipeline with all the fixes:
- ✅ **Epsilon threshold** instead of >0 for PICRUSt2 small positive values
- ✅ **Quantile-based pseudocount** with safety bounds (5th percentile/2, bounded between 1e-8 and 1e-4)
- ✅ **Aitchison variance** for compositionally-appropriate variance filtering
- ✅ **CLR validation** with zero-sum property and max absolute column sum reporting  
- ✅ **Flexible feature column** detection (function/pathway/etc.)
- ✅ **Proper feature counting** from abundance matrix, not total DataFrame
- ✅ **Kept features saving** to files for consistency across EC/KO/MetaCyc datasets

# Quality Control & Preprocessing

This section implements comprehensive quality control and preprocessing steps for microbiome functional data:

## QC & Preprocessing Steps:
1. **Feature Prevalence Filtering**: Remove features present in <20% of samples
2. **Low Variance Filtering**: Remove features with minimal variation across samples
3. **Compositional Data Transformation**: Apply CLR (Centered Log-Ratio) transformation
4. **Pseudocount Addition**: Handle zero values appropriately for log transformations
5. **Data Quality Assessment**: Evaluate filtering effects and transformation results

## Key Considerations:
- **Compositionality**: Microbiome data are compositional (relative abundances sum to 1)
- **Zero Inflation**: Many features have zero abundances in many samples
- **CLR Transformation**: Accounts for compositional nature and enables standard statistical methods
- **Prevalence Thresholds**: Balance between retaining informative features and reducing noise

### QC preprocessing Functions

In [ ]:
# QC & Preprocessing Functions with Type Safety and Robustness
import scipy.stats as stats
import numpy as np
import pandas as pd

def safe_numeric_conversion(data_matrix, fill_value=0.0):
    """
    Safely convert data to numeric types with comprehensive error handling
    
    Parameters:
    -----------
    data_matrix : pd.DataFrame
        Input data matrix
    fill_value : float
        Value to fill NaNs and infinities
        
    Returns:
    --------
    pd.DataFrame : Safely converted numeric data
    """
    # Convert to numeric, handling errors gracefully
    numeric_data = data_matrix.apply(pd.to_numeric, errors='coerce')
    
    # Replace infinities with NaN, then fill NaNs
    numeric_data = numeric_data.replace([np.inf, -np.inf], np.nan).fillna(fill_value)
    
    # Ensure all values are finite and non-negative for microbiome data
    numeric_data = numeric_data.clip(lower=0.0)
    
    return numeric_data

def calculate_feature_prevalence(data_matrix, epsilon=1e-10):
    """
    Calculate prevalence (percentage of samples with abundance > epsilon) for each feature
    Fixed to feature_axis=0 (features as rows) for PICRUSt2 consistency
    
    Parameters:
    -----------
    data_matrix : pd.DataFrame
        Data matrix (features as rows, samples as columns)
    epsilon : float
        Threshold for non-zero detection (for PICRUSt2 small positive values)
    
    Returns:
    --------
    pd.Series : Prevalence values for each feature
    """
    # Type safety: ensure numeric data
    data_numeric = safe_numeric_conversion(data_matrix)
    
    # Features are rows (fixed axis for consistency)
    prevalence = (data_numeric > epsilon).sum(axis=1) / data_numeric.shape[1] * 100
    
    return prevalence

def calculate_aitchison_variance(data_matrix, pseudocount=None):
    """
    Calculate Aitchison variance (variance of CLR-transformed data) for compositional data
    Fixed to feature_axis=0 with robust pseudocount calculation
    
    Parameters:
    -----------
    data_matrix : pd.DataFrame
        Data matrix (features as rows, samples as columns)
    pseudocount : float or None
        Pseudocount to add before CLR transformation
        
    Returns:
    --------
    pd.Series : Aitchison variance values for each feature
    """
    # Type safety: ensure numeric data
    data_numeric = safe_numeric_conversion(data_matrix)
    
    # Calculate robust pseudocount if not provided
    if pseudocount is None:
        # Extract actual positive values (flatten and remove zeros/NaNs)
        positive_values = data_numeric.values.flatten()
        positive_values = positive_values[positive_values > 0]
        positive_values = positive_values[~np.isnan(positive_values)]
        
        if len(positive_values) > 0:
            # Use quantile-based pseudocount with safety bounds
            q1_value = np.nanpercentile(positive_values, 1)  # 1st percentile
            q5_value = np.nanpercentile(positive_values, 5)  # 5th percentile
            
            # Robust pseudocount: 1st percentile / 2, bounded between 1e-9 and 1e-3
            pseudocount = np.clip(
                q1_value / 2, 
                a_min=1e-9,  # Lower bound
                a_max=1e-3   # Upper bound
            )
        else:
            pseudocount = 1e-8
    
    # Add pseudocount for CLR transformation
    data_for_clr = data_numeric + pseudocount
    
    # Apply CLR transformation with error handling
    try:
        log_data = np.log(data_for_clr)
        # Features are rows, samples are columns
        geometric_means = log_data.mean(axis=0)  # Geometric mean for each sample
        clr_data = log_data.subtract(geometric_means, axis=1)
        
        # Calculate Aitchison variance (variance of CLR values for each feature)
        aitchison_variance = clr_data.var(axis=1)
        
        # Handle NaN/inf in variance calculation
        aitchison_variance = aitchison_variance.replace([np.inf, -np.inf], np.nan)
        aitchison_variance = aitchison_variance.fillna(0.0)
        
    except Exception as e:
        print(f"  Warning: CLR calculation failed ({str(e)}), using standard variance")
        aitchison_variance = data_numeric.var(axis=1).fillna(0.0)
    
    return aitchison_variance

def get_flexible_feature_col(df):
    """
    Flexibly determine the feature column name across different datasets
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
        
    Returns:
    --------
    str or None : Feature column name
    """
    # Priority order for feature column detection
    possible_names = ['function', 'pathway', 'feature', 'EC', 'KO', 'METACYC']
    
    for name in possible_names:
        if name in df.columns:
            return name
    
    # If none found, return the first column if it's non-numeric
    if len(df.columns) > 0:
        first_col = df.columns[0]
        if not pd.api.types.is_numeric_dtype(df[first_col]):
            return first_col
    
    return None

def filter_features_by_prevalence(data_df, min_prevalence=20.0,
                                 feature_col=None, description_col='description',
                                 epsilon=1e-10):
    """
    Filter features based on prevalence threshold with robust error handling
    Fixed to feature_axis=0 (features as rows) for consistency
    
    Parameters:
    -----------
    data_df : pd.DataFrame
        Data with features and sample abundances
    min_prevalence : float
        Minimum prevalence percentage (0-100)
    feature_col : str or None
        Column name containing feature identifiers (auto-detected if None)
    description_col : str
        Column name containing feature descriptions
    epsilon : float
        Threshold for non-zero detection (for PICRUSt2 small positive values)
    
    Returns:
    --------
    tuple : (filtered_data, filter_stats, kept_features)
    """
    
    print(f"Filtering features by prevalence (>={min_prevalence}%)...")
    
    # Auto-detect feature column
    if feature_col is None:
        feature_col = get_flexible_feature_col(data_df)
    
    # Separate metadata columns from abundance data
    meta_cols = []
    if feature_col is not None and feature_col in data_df.columns:
        meta_cols.append(feature_col)
    if description_col in data_df.columns:
        meta_cols.append(description_col)
    
    if meta_cols:
        abundance_cols = [col for col in data_df.columns if col not in meta_cols]
        abundance_data = data_df[abundance_cols]
        meta_data = data_df[meta_cols]
    else:
        abundance_data = data_df
        meta_data = None
    
    # Type safety: ensure numeric abundance data
    abundance_data = safe_numeric_conversion(abundance_data)
    
    # Count original features (rows in abundance matrix)
    original_feature_count = len(abundance_data)
    
    # Calculate prevalence with error handling
    try:
        prevalence = calculate_feature_prevalence(abundance_data, epsilon=epsilon)
    except Exception as e:
        print(f"  Warning: Prevalence calculation error ({str(e)}), using zeros")
        prevalence = pd.Series(0.0, index=abundance_data.index)
    
    # Filter features with NaN-safe comparison
    features_to_keep = (prevalence >= min_prevalence) & (~prevalence.isna())
    kept_features = prevalence[features_to_keep].index.tolist()
    
    # Apply filter
    if meta_data is not None:
        filtered_data = pd.concat([
            meta_data[features_to_keep],
            abundance_data[features_to_keep]
        ], axis=1)
    else:
        filtered_data = abundance_data[features_to_keep]
    
    # Count filtered features
    filtered_feature_count = len(filtered_data)
    
    # Create filter statistics with safe calculations
    filter_stats = {
        'original_features': original_feature_count,
        'filtered_features': filtered_feature_count,
        'removed_features': original_feature_count - filtered_feature_count,
        'removal_percentage': ((original_feature_count - filtered_feature_count) / max(original_feature_count, 1)) * 100,
        'min_prevalence_threshold': min_prevalence,
        'epsilon_threshold': epsilon,
        'prevalence_distribution': {
            'mean': float(prevalence.mean()) if not prevalence.empty else 0.0,
            'median': float(prevalence.median()) if not prevalence.empty else 0.0,
            'min': float(prevalence.min()) if not prevalence.empty else 0.0,
            'max': float(prevalence.max()) if not prevalence.empty else 0.0,
            'q25': float(prevalence.quantile(0.25)) if not prevalence.empty else 0.0,
            'q75': float(prevalence.quantile(0.75)) if not prevalence.empty else 0.0
        }
    }
    
    print(f"  Original features: {filter_stats['original_features']}")
    print(f"  Features after prevalence filtering: {filter_stats['filtered_features']}")
    print(f"  Removed features: {filter_stats['removed_features']} ({filter_stats['removal_percentage']:.1f}%)")
    
    return filtered_data, filter_stats, kept_features

def filter_features_by_variance(data_df, min_variance_percentile=10.0,
                               feature_col=None, description_col='description'):
    """
    Filter features based on Aitchison variance threshold with robust error handling
    Fixed to feature_axis=0 (features as rows) for consistency
    
    Parameters:
    -----------
    data_df : pd.DataFrame
        Data with features and sample abundances
    min_variance_percentile : float
        Minimum variance percentile threshold (0-100)
    feature_col : str or None
        Column name containing feature identifiers (auto-detected if None)
    description_col : str
        Column name containing feature descriptions
    
    Returns:
    --------
    tuple : (filtered_data, filter_stats, kept_features)
    """
    
    print(f"Filtering features by Aitchison variance (>={min_variance_percentile}th percentile)...")
    
    # Auto-detect feature column
    if feature_col is None:
        feature_col = get_flexible_feature_col(data_df)
    
    # Separate metadata columns from abundance data
    meta_cols = []
    if feature_col is not None and feature_col in data_df.columns:
        meta_cols.append(feature_col)
    if description_col in data_df.columns:
        meta_cols.append(description_col)
    
    if meta_cols:
        abundance_cols = [col for col in data_df.columns if col not in meta_cols]
        abundance_data = data_df[abundance_cols]
        meta_data = data_df[meta_cols]
    else:
        abundance_data = data_df
        meta_data = None
    
    # Type safety: ensure numeric abundance data
    abundance_data = safe_numeric_conversion(abundance_data)
    
    # Count original features (rows in abundance matrix)
    original_feature_count = len(abundance_data)
    
    # Calculate Aitchison variance with error handling
    try:
        variance_values = calculate_aitchison_variance(abundance_data)
    except Exception as e:
        print(f"  Warning: Variance calculation error ({str(e)}), using standard variance")
        variance_values = abundance_data.var(axis=1).fillna(0.0)
    
    # Calculate threshold with NaN-safe percentile
    try:
        # Remove NaNs before percentile calculation
        valid_variances = variance_values.dropna()
        if len(valid_variances) > 0:
            variance_threshold = np.nanpercentile(valid_variances.values, min_variance_percentile)
        else:
            variance_threshold = 0.0
    except Exception as e:
        print(f"  Warning: Percentile calculation error ({str(e)}), using 0.0 threshold")
        variance_threshold = 0.0
    
    # Filter features with NaN-safe comparison
    features_to_keep = (variance_values >= variance_threshold) & (~variance_values.isna())
    kept_features = variance_values[features_to_keep].index.tolist()
    
    # Apply filter
    if meta_data is not None:
        filtered_data = pd.concat([
            meta_data[features_to_keep],
            abundance_data[features_to_keep]
        ], axis=1)
    else:
        filtered_data = abundance_data[features_to_keep]
    
    # Count filtered features
    filtered_feature_count = len(filtered_data)
    
    # Create filter statistics with safe calculations
    filter_stats = {
        'original_features': original_feature_count,
        'filtered_features': filtered_feature_count,
        'removed_features': original_feature_count - filtered_feature_count,
        'removal_percentage': ((original_feature_count - filtered_feature_count) / max(original_feature_count, 1)) * 100,
        'variance_threshold': float(variance_threshold),
        'variance_percentile': min_variance_percentile,
        'variance_type': 'aitchison',
        'variance_distribution': {
            'mean': float(variance_values.mean()) if not variance_values.empty else 0.0,
            'median': float(variance_values.median()) if not variance_values.empty else 0.0,
            'min': float(variance_values.min()) if not variance_values.empty else 0.0,
            'max': float(variance_values.max()) if not variance_values.empty else 0.0,
            'q25': float(variance_values.quantile(0.25)) if not variance_values.empty else 0.0,
            'q75': float(variance_values.quantile(0.75)) if not variance_values.empty else 0.0
        }
    }
    
    print(f"  Original features: {filter_stats['original_features']}")
    print(f"  Features after variance filtering: {filter_stats['filtered_features']}")
    print(f"  Removed features: {filter_stats['removed_features']} ({filter_stats['removal_percentage']:.1f}%)")
    print(f"  Aitchison variance threshold: {variance_threshold:.4f}")
    
    return filtered_data, filter_stats, kept_features

print("QC & Preprocessing functions loaded successfully!")
print("Available functions:")
print("- safe_numeric_conversion(): Type-safe numeric conversion with inf/NaN handling")
print("- calculate_feature_prevalence(): Calculate feature prevalence with epsilon threshold")
print("- calculate_aitchison_variance(): Calculate compositionally-appropriate variance")
print("- get_flexible_feature_col(): Auto-detect feature column (function/pathway/etc)")
print("- filter_features_by_prevalence(): Filter features by minimum prevalence")
print("- filter_features_by_variance(): Filter features by Aitchison variance")
print("- All functions use feature_axis=0 (features as rows) for PICRUSt2 consistency")

### CLR transformation functions

In [ ]:
# CLR Transformation Functions with Type Safety and Robust Pseudocount
from sklearn.preprocessing import LabelEncoder
import numpy as np
import pandas as pd

def add_pseudocount(data_matrix, pseudocount=None, method='quantile_bound'):
    """
    Add pseudocount to zero values for CLR transformation with quantile-based bounds
    Fixed to feature_axis=0 for consistency
    
    Parameters:
    -----------
    data_matrix : pd.DataFrame
        Data matrix (features as rows, samples as columns)
    pseudocount : float or None
        Specific pseudocount value. If None, calculated automatically
    method : str
        Method for calculating pseudocount ('quantile_bound', 'adaptive', 'fixed')
        
    Returns:
    --------
    tuple : (data_with_pseudocount, pseudocount_used, stats)
    """
    # Type safety: ensure numeric data
    data_numeric = safe_numeric_conversion(data_matrix)
    
    if pseudocount is None:
        # Extract all positive values for quantile calculation
        positive_values = data_numeric.values.flatten()
        positive_values = positive_values[positive_values > 0]
        positive_values = positive_values[~np.isnan(positive_values)]
        
        if len(positive_values) == 0:
            pseudocount = 1e-6
            print("  Warning: No positive values found, using default pseudocount 1e-6")
        else:
            if method == 'quantile_bound':
                # Quantile-based with safety bounds
                q1_value = np.nanpercentile(positive_values, 1)  # 1st percentile
                q5_value = np.nanpercentile(positive_values, 5)  # 5th percentile
                
                # Robust pseudocount: 1st percentile / 2, with bounds
                pseudocount = np.clip(
                    q1_value / 2,
                    a_min=1e-9,   # Lower bound to prevent underflow
                    a_max=1e-3    # Upper bound to prevent overestimation
                )
                
            elif method == 'adaptive':
                # Adaptive based on data range
                min_pos = np.nanmin(positive_values)
                pseudocount = np.clip(min_pos / 10, 1e-9, 1e-4)
                
            else:  # 'fixed'
                pseudocount = 1e-6
    
    # Apply pseudocount only to zero/negative values
    data_with_pseudocount = data_numeric.copy()
    zero_mask = (data_numeric <= 0) | data_numeric.isna()
    data_with_pseudocount[zero_mask] = pseudocount
    
    # Calculate statistics
    n_zeros_replaced = zero_mask.sum().sum()
    total_values = data_numeric.size
    replacement_percentage = (n_zeros_replaced / total_values) * 100
    
    stats = {
        'method': method,
        'pseudocount_value': float(pseudocount),
        'zeros_replaced': int(n_zeros_replaced),
        'total_values': int(total_values),
        'replacement_percentage': float(replacement_percentage),
        'min_original': float(data_numeric.min().min()) if not data_numeric.empty else np.nan,
        'min_after_pseudocount': float(data_with_pseudocount.min().min()) if not data_with_pseudocount.empty else np.nan
    }
    
    print(f"  Pseudocount applied: {pseudocount:.2e} (method: {method})")
    print(f"  Values replaced: {n_zeros_replaced}/{total_values} ({replacement_percentage:.1f}%)")
    
    return data_with_pseudocount, pseudocount, stats

def clr_transform(data_df, feature_col=None, description_col='description', 
                 pseudocount=None, validate_transform=True):
    """
    Apply Centered Log-Ratio (CLR) transformation to compositional data
    Fixed to feature_axis=0 with comprehensive type safety and validation
    
    Parameters:
    -----------
    data_df : pd.DataFrame
        Data with features and sample abundances
    feature_col : str or None
        Column name containing feature identifiers (auto-detected if None)
    description_col : str
        Column name containing feature descriptions
    pseudocount : float or None
        Pseudocount for zero handling (auto-calculated if None)
    validate_transform : bool
        Whether to validate the CLR transformation
        
    Returns:
    --------
    tuple : (clr_data, transform_stats)
    """
    
    print("Applying CLR transformation...")
    
    # Auto-detect feature column
    if feature_col is None:
        feature_col = get_flexible_feature_col(data_df)
    
    # Separate metadata columns from abundance data
    meta_cols = []
    if feature_col is not None and feature_col in data_df.columns:
        meta_cols.append(feature_col)
    if description_col in data_df.columns:
        meta_cols.append(description_col)
    
    if meta_cols:
        abundance_cols = [col for col in data_df.columns if col not in meta_cols]
        abundance_data = data_df[abundance_cols]
        meta_data = data_df[meta_cols]
    else:
        abundance_data = data_df
        meta_data = None
    
    # Type safety: ensure numeric abundance data
    abundance_data = safe_numeric_conversion(abundance_data)
    
    # Add pseudocount with robust calculation
    try:
        data_with_pseudocount, pseudocount_used, pseudocount_stats = add_pseudocount(
            abundance_data, pseudocount=pseudocount, method='quantile_bound'
        )
    except Exception as e:
        print(f"  Warning: Pseudocount calculation failed ({str(e)}), using fixed value")
        pseudocount_used = 1e-6 if pseudocount is None else pseudocount
        data_with_pseudocount = abundance_data.copy()
        data_with_pseudocount[data_with_pseudocount <= 0] = pseudocount_used
        pseudocount_stats = {'method': 'fallback', 'pseudocount_value': pseudocount_used}
    
    # Apply CLR transformation with error handling
    try:
        # Calculate log of data
        log_data = np.log(data_with_pseudocount)
        
        # Calculate geometric mean for each sample (axis=0 for samples as columns)
        geometric_means = log_data.mean(axis=0)
        
        # Subtract geometric mean from each feature's log values
        clr_data_values = log_data.subtract(geometric_means, axis=1)
        
        # Create CLR dataframe with original structure
        if meta_data is not None:
            clr_data = pd.concat([meta_data, clr_data_values], axis=1)
        else:
            clr_data = clr_data_values
            
    except Exception as e:
        print(f"  Error: CLR transformation failed: {str(e)}")
        return data_df, {'error': str(e)}
    
    # Validation of CLR transformation
    if validate_transform:
        try:
            # Check if geometric means are approximately zero (CLR property)
            clr_geometric_means = clr_data_values.mean(axis=0)
            max_deviation = np.abs(clr_geometric_means).max()
            
            if max_deviation > 1e-10:
                print(f"  Warning: CLR validation failed, max deviation: {max_deviation:.2e}")
            else:
                print(f"  CLR validation passed (max deviation: {max_deviation:.2e})")
                
        except Exception as e:
            print(f"  Warning: CLR validation error: {str(e)}")
    
    # Calculate transformation statistics
    try:
        transform_stats = {
            'original_shape': abundance_data.shape,
            'clr_shape': clr_data_values.shape,
            'pseudocount_stats': pseudocount_stats,
            'clr_data_range': {
                'min': float(clr_data_values.min().min()) if not clr_data_values.empty else np.nan,
                'max': float(clr_data_values.max().max()) if not clr_data_values.empty else np.nan,
                'mean': float(clr_data_values.mean().mean()) if not clr_data_values.empty else np.nan,
                'std': float(clr_data_values.std().mean()) if not clr_data_values.empty else np.nan
            },
            'validation_passed': validate_transform and max_deviation <= 1e-10 if 'max_deviation' in locals() else None
        }
    except Exception as e:
        transform_stats = {'error': f'Stats calculation failed: {str(e)}'}
    
    print(f"  CLR transformation completed")
    print(f"  Shape: {abundance_data.shape} -> {clr_data_values.shape}")
    print(f"  Data range: [{transform_stats.get('clr_data_range', {}).get('min', 'N/A'):.3f}, {transform_stats.get('clr_data_range', {}).get('max', 'N/A'):.3f}]")
    
    return clr_data, transform_stats

def batch_clr_transform(data_dict, feature_col=None, description_col='description',
                       pseudocount=None, validate_transform=True):
    """
    Apply CLR transformation to multiple datasets with consistent parameters
    
    Parameters:
    -----------
    data_dict : dict
        Dictionary of {name: dataframe} pairs
    feature_col : str or None
        Column name containing feature identifiers (auto-detected if None)
    description_col : str
        Column name containing feature descriptions
    pseudocount : float or None
        Pseudocount for zero handling (auto-calculated if None)
    validate_transform : bool
        Whether to validate the CLR transformation
        
    Returns:
    --------
    tuple : (clr_data_dict, batch_stats)
    """
    
    print(f"Applying CLR transformation to {len(data_dict)} datasets...")
    
    clr_data_dict = {}
    batch_stats = {}
    
    for name, data_df in data_dict.items():
        print(f"\nProcessing {name}:")
        try:
            clr_data, transform_stats = clr_transform(
                data_df, 
                feature_col=feature_col,
                description_col=description_col,
                pseudocount=pseudocount,
                validate_transform=validate_transform
            )
            clr_data_dict[name] = clr_data
            batch_stats[name] = transform_stats
            
        except Exception as e:
            print(f"  Error processing {name}: {str(e)}")
            clr_data_dict[name] = data_df  # Keep original on error
            batch_stats[name] = {'error': str(e)}
    
    print(f"\nBatch CLR transformation completed for {len(clr_data_dict)} datasets")
    
    return clr_data_dict, batch_stats

print("CLR transformation functions loaded successfully!")
print("Available functions:")
print("- add_pseudocount(): Add quantile-bounded pseudocount to zero values")
print("- clr_transform(): Apply CLR transformation with type safety and validation")
print("- batch_clr_transform(): Apply CLR to multiple datasets consistently")
print("- All functions use feature_axis=0 (features as rows) for consistency")

### visualizations functions

In [ ]:
def create_feature_summary_plots(filter_stats_dict, save_plots=True, output_dir='plots'):
    """
    Create comprehensive summary plots for feature filtering results
    with ASCII-safe text output for headless environments
    
    Parameters:
    -----------
    filter_stats_dict : dict
        Dictionary of filter statistics from preprocessing steps
    save_plots : bool
        Whether to save plots to files
    output_dir : str
        Directory to save plots
        
    Returns:
    --------
    dict : Plot information and statistics
    """
    
    print("Creating feature filtering summary plots...")
    
    # Ensure output directory exists
    if save_plots:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Extract data for plotting with type safety
    plot_data = {}
    
    for dataset_name, stats in filter_stats_dict.items():
        if 'error' not in stats and stats is not None:
            # Safely extract numeric values
            try:
                original = int(stats.get('original_features', 0))
                filtered = int(stats.get('filtered_features', 0))
                removed = int(stats.get('removed_features', 0))
                removal_pct = float(stats.get('removal_percentage', 0.0))
                
                # Sanity check: original should equal removed + filtered
                if original != (removed + filtered):
                    print(f"  Warning: Sanity check failed for {dataset_name}: "
                          f"original({original}) != removed({removed}) + filtered({filtered})")
                
                plot_data[dataset_name] = {
                    'original': original,
                    'filtered': filtered,
                    'removed': removed,
                    'removal_pct': removal_pct,
                    'prevalence_threshold': stats.get('min_prevalence_threshold', None),
                    'variance_threshold': stats.get('variance_threshold', None),
                    'variance_percentile': stats.get('variance_percentile', None)
                }
            except (ValueError, TypeError) as e:
                print(f"  Warning: Data extraction error for {dataset_name}: {str(e)}")
                continue
    
    if not plot_data:
        print("  No valid data for plotting")
        return {'error': 'No valid data', 'plot_saved': False}
    
    # Create figure with multiple subplots (2x3 to separate threshold types)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Feature Filtering Summary', fontsize=16, fontweight='bold')
    
    # Plot 1: Feature counts (before/after)
    datasets = list(plot_data.keys())
    original_counts = [plot_data[d]['original'] for d in datasets]
    filtered_counts = [plot_data[d]['filtered'] for d in datasets]
    
    x_pos = np.arange(len(datasets))
    width = 0.35
    
    axes[0, 0].bar(x_pos - width/2, original_counts, width, 
                   label='Original', alpha=0.8, color='skyblue')
    axes[0, 0].bar(x_pos + width/2, filtered_counts, width, 
                   label='After Filtering', alpha=0.8, color='lightcoral')
    
    axes[0, 0].set_xlabel('Datasets')
    axes[0, 0].set_ylabel('Feature Count')
    axes[0, 0].set_title('Feature Counts: Before vs After Filtering')
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(datasets, rotation=45, ha='right')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Removal percentages
    removal_pcts = [plot_data[d]['removal_pct'] for d in datasets]
    
    bars = axes[0, 1].bar(datasets, removal_pcts, color='orange', alpha=0.7)
    axes[0, 1].set_xlabel('Datasets')
    axes[0, 1].set_ylabel('Removal Percentage (%)')
    axes[0, 1].set_title('Feature Removal Percentages')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Add percentage labels on bars (using bar_label if available, fallback to annotate)
    try:
        # Modern matplotlib (>= 3.4)
        axes[0, 1].bar_label(bars, labels=[f'{pct:.1f}%' for pct in removal_pcts], 
                            padding=3, fontsize=9)
    except AttributeError:
        # Fallback for older matplotlib
        for bar, pct in zip(bars, removal_pcts):
            height = bar.get_height()
            axes[0, 1].annotate(f'{pct:.1f}%',
                               xy=(bar.get_x() + bar.get_width() / 2, height),
                               xytext=(0, 3),  # 3 points vertical offset
                               textcoords="offset points",
                               ha='center', va='bottom', fontsize=9)
    
    # Plot 3: Feature reduction overview (CORRECTED stacking)
    removed_counts = [plot_data[d]['removed'] for d in datasets]
    
    # First bar: kept features (no bottom)
    axes[0, 2].bar(datasets, filtered_counts, label='Kept Features', 
                   color='lightgreen', alpha=0.8)
    # Second bar: removed features stacked ON TOP of kept features
    axes[0, 2].bar(datasets, removed_counts, bottom=filtered_counts,
                   label='Removed Features', color='lightcoral', alpha=0.8)
    
    axes[0, 2].set_xlabel('Datasets')
    axes[0, 2].set_ylabel('Feature Count')
    axes[0, 2].set_title('Feature Composition: Kept vs Removed')
    axes[0, 2].tick_params(axis='x', rotation=45)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Plot 4: Prevalence thresholds (SEPARATED)
    prevalence_thresholds = [plot_data[d]['prevalence_threshold'] 
                           for d in datasets if plot_data[d]['prevalence_threshold'] is not None]
    prev_datasets = [d for d in datasets if plot_data[d]['prevalence_threshold'] is not None]
    
    if prevalence_thresholds and any(t > 0 for t in prevalence_thresholds):
        axes[1, 0].bar(prev_datasets, prevalence_thresholds, color='blue', alpha=0.7)
        axes[1, 0].set_xlabel('Datasets')
        axes[1, 0].set_ylabel('Prevalence Threshold (%)')
        axes[1, 0].set_title('Prevalence Filtering Thresholds')
        axes[1, 0].tick_params(axis='x', rotation=45)
        axes[1, 0].grid(True, alpha=0.3)
    else:
        axes[1, 0].text(0.5, 0.5, 'No prevalence threshold data', 
                       ha='center', va='center', transform=axes[1, 0].transAxes,
                       fontsize=12, style='italic')
        axes[1, 0].set_title('Prevalence Filtering Thresholds')
    
    # Plot 5: Variance thresholds (SEPARATED)
    variance_thresholds = [plot_data[d]['variance_threshold'] 
                         for d in datasets if plot_data[d]['variance_threshold'] is not None]
    var_datasets = [d for d in datasets if plot_data[d]['variance_threshold'] is not None]
    
    if variance_thresholds and any(t > 0 for t in variance_thresholds):
        axes[1, 1].bar(var_datasets, variance_thresholds, color='red', alpha=0.7)
        axes[1, 1].set_xlabel('Datasets')
        axes[1, 1].set_ylabel('Aitchison Variance Threshold')
        axes[1, 1].set_title('Variance Filtering Thresholds')
        axes[1, 1].tick_params(axis='x', rotation=45)
        axes[1, 1].grid(True, alpha=0.3)
    else:
        axes[1, 1].text(0.5, 0.5, 'No variance threshold data', 
                       ha='center', va='center', transform=axes[1, 1].transAxes,
                       fontsize=12, style='italic')
        axes[1, 1].set_title('Variance Filtering Thresholds')
    
    # Plot 6: Summary statistics
    axes[1, 2].text(0.05, 0.95, f"Summary Statistics:\n\n" +
                   f"Total datasets: {len(datasets)}\n" +
                   f"Avg removal: {np.mean(removal_pcts):.1f}%\n" +
                   f"Total original features: {sum(original_counts):,}\n" +
                   f"Total final features: {sum(filtered_counts):,}\n" +
                   f"Total removed: {sum(removed_counts):,}",
                   transform=axes[1, 2].transAxes, verticalalignment='top',
                   fontfamily='monospace', fontsize=10,
                   bbox=dict(boxstyle="round,pad=0.5", facecolor='lightgray', alpha=0.8))
    axes[1, 2].set_xlim(0, 1)
    axes[1, 2].set_ylim(0, 1)
    axes[1, 2].axis('off')
    axes[1, 2].set_title('Summary Statistics')
    
    plt.tight_layout()
    
    # Save plot if requested
    plot_save_success = False
    if save_plots:
        try:
            plot_path = Path(output_dir) / 'feature_filtering_summary.png'
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            print(f"  Plot saved: {plot_path}")
            plot_save_success = True
        except Exception as e:
            print(f"  Warning: Could not save plot: {str(e)}")
            plot_save_success = False
    
    plt.show()
    
    # Free memory
    plt.close(fig)
    
    # Print ASCII-safe summary
    print("\n" + "="*60)
    print("FEATURE FILTERING RESULTS SUMMARY")
    print("="*60)
    
    for dataset_name, data in plot_data.items():
        print(f"\n{dataset_name}:")
        print(f"  Original features: {data['original']:,}")
        print(f"  Features after filtering: {data['filtered']:,}")
        print(f"  Features removed: {data['removed']:,} ({data['removal_pct']:.1f}%)")
        if data['prevalence_threshold']:
            print(f"  Prevalence threshold: {data['prevalence_threshold']:.1f}%")
        if data['variance_threshold']:
            print(f"  Variance threshold: {data['variance_threshold']:.4f}")
    
    print("="*60)
    
    return {
        'plot_data': plot_data,
        'total_datasets': len(datasets),
        'average_removal_pct': np.mean(removal_pcts) if removal_pcts else 0.0,
        'plot_saved': plot_save_success  # CORRECTED: actual save status
    }


### Core function

In [ ]:
# Visualization and Analysis Execution Functions with Type Safety
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
# Fix in execute_comprehensive_preprocessing function around line 2690
def get_feature_count(data_df):
    """Correctly count features regardless of dataset type"""
    # Check for metadata columns that indicate features are in rows
    metadata_indicators = ['function', 'pathway', 'description', 'ASV_ID']
    has_metadata_cols = any(col in data_df.columns for col in metadata_indicators)
    
    if has_metadata_cols:
        # Features are rows, samples are columns (exclude metadata columns)
        sample_cols = [col for col in data_df.columns 
                      if col not in ['function', 'pathway', 'description', 'ASV_ID']]
        return data_df.shape[0], len(sample_cols)  # (features, samples)
    else:
        # Assume all columns are samples (like transposed ASV table)
        return data_df.shape[1], data_df.shape[0]  # (features, samples)

def execute_comprehensive_preprocessing(data_dict, 
                                      min_prevalence=20.0,
                                      min_variance_percentile=10.0,
                                      apply_clr=True,
                                      create_plots=True,
                                      save_results=True,
                                      output_dir='processed_data',
                                      epsilon=1e-10,
                                      pseudocount=None,
                                      random_state=42):
    """
    Execute comprehensive preprocessing pipeline with type safety and error handling
    
    Parameters:
    -----------
    data_dict : dict
        Dictionary of {dataset_name: dataframe} pairs
    min_prevalence : float
        Minimum prevalence percentage for filtering
    min_variance_percentile : float
        Minimum variance percentile for filtering
    apply_clr : bool
        Whether to apply CLR transformation
    create_plots : bool
        Whether to create summary plots
    save_results : bool
        Whether to save processed datasets
    output_dir : str
        Directory for saving results
    epsilon : float
        Threshold for non-zero detection (for PICRUSt2 small positive values)
    pseudocount : float or None
        Pseudocount for CLR transformation
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    dict : Complete preprocessing results
    """
    
    print("\n" + "="*80)
    print("COMPREHENSIVE MICROBIOME DATA PREPROCESSING PIPELINE")
    print("="*80)
    print(f"Processing {len(data_dict)} datasets...")
    print(f"Prevalence threshold: {min_prevalence}% (epsilon: {epsilon})")
    print(f"Variance percentile threshold: {min_variance_percentile}%")
    print(f"CLR transformation: {'Enabled' if apply_clr else 'Disabled'}")
    print(f"Random state: {random_state}")
    print("="*80)
    
    # Initialize result containers
    results = {
        'original_data': data_dict.copy(),
        'prevalence_filtered': {},
        'variance_filtered': {},
        'clr_transformed': {} if apply_clr else None,
        'filter_stats': {},
        'clr_stats': {} if apply_clr else None,
        'processing_summary': {}
    }
    
    # Ensure output directory exists
    if save_results:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    # Step 1: Prevalence filtering
    print(f"\nStep 1: Prevalence Filtering (>= {min_prevalence}%)")
    print("-" * 50)
    
    prevalence_stats = {}
    
    for dataset_name, data_df in data_dict.items():
        print(f"\nProcessing {dataset_name}:")
        
        # Type safety check
        if not isinstance(data_df, pd.DataFrame) or data_df.empty:
            print(f"  Warning: Invalid data for {dataset_name}, skipping")
            prevalence_stats[dataset_name] = {'error': 'Invalid data', 'processing_successful': False}
            continue
        
        try:
            # CORRECTED: Pass epsilon parameter
            filtered_data, filter_stats, kept_features = filter_features_by_prevalence(
                data_df, 
                min_prevalence=min_prevalence,
                feature_col=None,  # Auto-detect
                epsilon=epsilon  # ADDED: epsilon parameter
            )
            
            results['prevalence_filtered'][dataset_name] = filtered_data
            filter_stats['processing_successful'] = True
            prevalence_stats[dataset_name] = filter_stats
            
        except Exception as e:
            print(f"  Error in prevalence filtering for {dataset_name}: {str(e)}")
            results['prevalence_filtered'][dataset_name] = data_df  # Keep original
            prevalence_stats[dataset_name] = {'error': str(e), 'processing_successful': False}
    
    # Step 2: Variance filtering
    print(f"\nStep 2: Variance Filtering (>= {min_variance_percentile}th percentile)")
    print("-" * 50)
    
    variance_stats = {}
    
    for dataset_name, data_df in results['prevalence_filtered'].items():
        print(f"\nProcessing {dataset_name}:")
        
        try:
            # CORRECTED: Pass pseudocount if function expects it
            filtered_data, filter_stats, kept_features = filter_features_by_variance(
                data_df,
                min_variance_percentile=min_variance_percentile,
                feature_col=None  # Auto-detect
            )
            
            results['variance_filtered'][dataset_name] = filtered_data
            filter_stats['processing_successful'] = True
            variance_stats[dataset_name] = filter_stats
            
        except Exception as e:
            print(f"  Error in variance filtering for {dataset_name}: {str(e)}")
            results['variance_filtered'][dataset_name] = data_df  # Keep original
            variance_stats[dataset_name] = {'error': str(e), 'processing_successful': False}
    
    # Combine filter statistics
    results['filter_stats'] = {
        'prevalence': prevalence_stats,
        'variance': variance_stats
    }
    
    # Step 3: CLR transformation (if requested)
    if apply_clr:
        print(f"\nStep 3: CLR Transformation")
        print("-" * 50)
        
        # CORRECTED: Pass pseudocount and random_state
        clr_data_dict, clr_stats = batch_clr_transform(
            results['variance_filtered'],
            feature_col=None,  # Auto-detect
            pseudocount=pseudocount,  # ADDED: pseudocount parameter
            validate_transform=True
        )
        
        results['clr_transformed'] = clr_data_dict
        results['clr_stats'] = clr_stats
    
    # Step 4: Create summary plots (if requested)
    if create_plots:
        print(f"\nStep 4: Creating Summary Plots")
        print("-" * 50)
        
        try:
            # CORRECTED: Only include datasets without errors
            combined_stats = {}
            for dataset_name in data_dict.keys():
                if (dataset_name in variance_stats and 
                    'error' not in variance_stats[dataset_name] and
                    variance_stats[dataset_name].get('processing_successful', False)):
                    # Use variance stats as they represent the final filtering step
                    combined_stats[dataset_name] = variance_stats[dataset_name]
            
            if combined_stats:
                plot_results = create_feature_summary_plots(
                    combined_stats,
                    save_plots=save_results,
                    output_dir=output_dir
                )
                results['plot_results'] = plot_results
            else:
                print("  No successful filtering results to plot")
                results['plot_results'] = {'error': 'No successful results', 'plot_saved': False}
            
        except Exception as e:
            print(f"  Warning: Plot creation failed: {str(e)}")
            results['plot_results'] = {'error': str(e), 'plot_saved': False}
    
    # Step 5: Save processed data (if requested)
    if save_results:
        print(f"\nStep 5: Saving Processed Data")
        print("-" * 50)
        
        save_summary = {}
        
        # Save variance-filtered data
        for dataset_name, data_df in results['variance_filtered'].items():
            try:
                output_path = Path(output_dir) / f"{dataset_name}_variance_filtered.tsv"
                data_df.to_csv(output_path, sep='\t', index=False)
                save_summary[f"{dataset_name}_variance_filtered"] = str(output_path)
                print(f"  Saved: {output_path}")
            except Exception as e:
                print(f"  Warning: Could not save {dataset_name}_variance_filtered: {str(e)}")
        
        # Save CLR-transformed data (if available)
        if apply_clr and results['clr_transformed']:
            for dataset_name, data_df in results['clr_transformed'].items():
                try:
                    output_path = Path(output_dir) / f"{dataset_name}_clr_transformed.tsv"
                    data_df.to_csv(output_path, sep='\t', index=False)
                    save_summary[f"{dataset_name}_clr_transformed"] = str(output_path)
                    print(f"  Saved: {output_path}")
                except Exception as e:
                    print(f"  Warning: Could not save {dataset_name}_clr_transformed: {str(e)}")
        
        results['saved_files'] = save_summary
    
    # Generate processing summary
    print(f"\nGenerating Processing Summary...")
    print("-" * 50)
    
    processing_summary = {}
    
    for dataset_name in data_dict.keys():
        try:
            # CORRECTED: Check if features are rows or columns
            data_df = data_dict[dataset_name]
            #is_row_features = ('function' in data_df.columns or 'pathway' in data_df.columns)
            #original_features = data_df.shape[0] if is_row_features else data_df.shape[1]
            
            original_features, original_samples = get_feature_count(data_dict[dataset_name])

            # Get filtered counts safely
            prevalence_data = results['prevalence_filtered'].get(dataset_name)
            variance_data = results['variance_filtered'].get(dataset_name)

            # For filtered data:
            if isinstance(prevalence_data, pd.DataFrame):
                prevalence_features, _ = get_feature_count(prevalence_data)
            else:
                prevalence_features = original_features

            if isinstance(variance_data, pd.DataFrame):
                variance_features, _ = get_feature_count(variance_data)
            else:
                variance_features = prevalence_features
                
            # Check processing success
            processing_successful = (
                variance_stats.get(dataset_name, {}).get('processing_successful', False) and
                prevalence_stats.get(dataset_name, {}).get('processing_successful', False)
            )
            
            summary = {
                'original_features': original_features,
                'after_prevalence_filter': prevalence_features,
                'after_variance_filter': variance_features,
                'total_features_removed': original_features - variance_features,
                'total_removal_percentage': ((original_features - variance_features) / max(original_features, 1)) * 100,
                'clr_applied': apply_clr and dataset_name in results.get('clr_transformed', {}),
                'processing_successful': processing_successful  # CORRECTED: Mark failed processing
            }
            
            processing_summary[dataset_name] = summary
            
        except Exception as e:
            print(f"  Warning: Summary calculation error for {dataset_name}: {str(e)}")
            processing_summary[dataset_name] = {'error': str(e), 'processing_successful': False}
    
    results['processing_summary'] = processing_summary
    
    # Print final summary
    print("\n" + "="*80)
    print("PREPROCESSING PIPELINE COMPLETED")
    print("="*80)
    
    for dataset_name, summary in processing_summary.items():
        if 'error' not in summary:
            print(f"\n{dataset_name}:")
            print(f"  Features: {summary['original_features']} -> {summary['after_variance_filter']}")
            print(f"  Removed: {summary['total_features_removed']} ({summary['total_removal_percentage']:.1f}%)")
            print(f"  CLR applied: {'Yes' if summary['clr_applied'] else 'No'}")
            print(f"  Status: {'Success' if summary['processing_successful'] else 'Failed'}")
        else:
            print(f"\n{dataset_name}: FAILED - {summary['error']}")
    
    print("="*80)
    
    return results

print("Visualization and execution functions loaded successfully!")
print("Available functions:")
print("- create_feature_summary_plots(): Create comprehensive filtering summary plots")
print("- execute_comprehensive_preprocessing(): Run complete preprocessing pipeline")
print("- All functions include type safety, error handling, and ASCII-safe output")
print("- Fixed: sanity checks, separated threshold plots, proper stacking, memory management")
print("- Added: epsilon, pseudocount, and random_state parameters for reproducibility")

### execution

In [ ]:
# Execute Preprocessing Pipeline on All Functional Datasets
print("🚀 EXECUTING PREPROCESSING PIPELINE ON ALL FUNCTIONAL DATASETS")
print("=" * 80)

# Check if clean_data exists, if not load from files
if 'clean_data' not in globals() or not clean_data:
    print("📂 Loading data from files...")
    
    # Load the functional datasets
    file_paths = {
        'ec_function':  f"{custom_outputdir}/analysis_ready_data/ec_function_clean.csv",
        'ko_function':  f"{custom_outputdir}/analysis_ready_data/ko_ec_function_clean.csv", 
        'metacyc_pathway':  f"{custom_outputdir}/analysis_ready_data/metacyc_ec_function_clean.csv'",
        'asv_table_transposed': f"{custom_outputdir}/analysis_ready_data/asv_table_transposed_clean.csv",
        'metadata': f"{custom_outputdir}/analysis_ready_data/metadata_clean.csv"
    }
    
    clean_data = {}
    for dataset_name, file_path in file_paths.items():
        try:
            print(f"  Loading {dataset_name} from {file_path}...")
            clean_data[dataset_name] = pd.read_csv(file_path, index_col=0)
            print(f"    Shape: {clean_data[dataset_name].shape}")
        except FileNotFoundError:
            print(f"    Warning: {file_path} not found, skipping {dataset_name}")
        except Exception as e:
            print(f"    Error loading {file_path}: {str(e)}")

# Prepare functional datasets for preprocessing
all_data_dict = {}

# Check available datasets and prepare them
if 'ec_function' in clean_data:
    all_data_dict['EC_Functions'] = clean_data['ec_function']
if 'ko_function' in clean_data:
    all_data_dict['KO_Functions'] = clean_data['ko_function'] 
if 'metacyc_pathway' in clean_data:
    all_data_dict['MetaCyc_Pathways'] = clean_data['metacyc_pathway']
if 'asv_table' in clean_data:
    all_data_dict['asv_table'] = clean_data['asv_table']

if not all_data_dict:
    print("❌ No functional datasets available for preprocessing!")
    print("Available keys in clean_data:", list(clean_data.keys()) if 'clean_data' in globals() else "clean_data not found")
else:
    print(f"✅ Found {len(all_data_dict)} functional datasets to process:")
    for name, data in all_data_dict.items():
        print(f"   • {name}: {data.shape}")

#for dataset_name, data_df in all_data_dict.items():
#    print(f"\nProcessing dataset, including valid PC_codes: {dataset_name}")
    # Filter columns to only include valid PC_codes (keep metadata columns)
#    print(data_df.columns)

#break
# Preprocessing parameters
preprocessing_params = {
    'min_prevalence': 15.0,           # Features must be present in ≥15% of samples
    'min_variance_percentile': 10.0,  # Remove bottom 10% of features by variance
    'apply_clr': True,                # Apply CLR transformation
    'create_plots': True,             # Create summary plots
    'save_results': True              # Save processed data
}

print(f"📋 Preprocessing Parameters:")
print(f"   • Minimum prevalence: ≥{preprocessing_params['min_prevalence']}% of samples")
print(f"   • Variance filter: ≥{preprocessing_params['min_variance_percentile']}th percentile")
print(f"   • CLR transformation: {'Enabled' if preprocessing_params['apply_clr'] else 'Disabled'}")
print(f"   • Create plots: {'Enabled' if preprocessing_params['create_plots'] else 'Disabled'}")
print(f"   • Save results: {'Enabled' if preprocessing_params['save_results'] else 'Disabled'}")

# Execute comprehensive preprocessing pipeline
if all_data_dict:
    preprocessing_results = execute_comprehensive_preprocessing(
        data_dict=all_data_dict,
        min_prevalence=preprocessing_params['min_prevalence'],
        min_variance_percentile=preprocessing_params['min_variance_percentile'],
        apply_clr=preprocessing_params['apply_clr'],
        create_plots=preprocessing_params['create_plots'],
        save_results=preprocessing_params['save_results'],
        output_dir=str(custom_outputdir / 'preprocessing_output')
    )
    
    print("\n" + "=" * 80)
    print("🎉 PREPROCESSING PIPELINE COMPLETED FOR ALL DATASETS!")
    print("=" * 80)
    
    # Enhanced summary with detailed statistics
    print("\n📊 DETAILED PREPROCESSING SUMMARY:")
    print("-" * 80)
    
    summary_data = []
    for dataset_name, summary in preprocessing_results['processing_summary'].items():
        if 'error' not in summary:
            summary_data.append({
                'Dataset': dataset_name,
                'Original Features': summary['original_features'],
                'After Prevalence': summary['after_prevalence_filter'],
                'Final Features': summary['after_variance_filter'],
                'Total Removed': summary['total_features_removed'],
                'Removal %': f"{summary['total_removal_percentage']:.1f}%",
                'CLR Applied': 'Yes' if summary['clr_applied'] else 'No',
                'Status': 'Success' if summary['processing_successful'] else 'Failed'
            })
    
    if summary_data:
        summary_df = pd.DataFrame(summary_data)
        print(summary_df.to_string(index=False))
        
        # Calculate overall statistics
        total_original = sum([d['Original Features'] for d in summary_data])
        total_final = sum([int(d['Final Features']) for d in summary_data])
        overall_removal_pct = ((total_original - total_final) / total_original) * 100 if total_original > 0 else 0
        
        print(f"\n📈 OVERALL STATISTICS:")
        print(f"   • Total original features: {total_original:,}")
        print(f"   • Total final features: {total_final:,}")
        print(f"   • Overall removal: {total_original - total_final:,} features ({overall_removal_pct:.1f}%)")
        
        # Output directory information
        output_path = custom_outputdir / 'preprocessing_output'
        print(f"\n📁 OUTPUT LOCATIONS:")
        print(f"   • Processed data: {output_path}")
        print(f"   • Summary plots: {output_path}")
        
        print("\n🔬 READY FOR DOWNSTREAM ANALYSIS!")
        print("   • Filtered features by prevalence and variance")
        print("   • Applied robust pseudocount with quantile bounds")
        print("   • CLR-transformed data ready for standard statistical methods") 
        print("   • Maintained compositionality principles throughout")
        print("   • Type-safe preprocessing with comprehensive error handling")
    else:
        print("❌ No successful preprocessing results to summarize")
    
    print("=" * 80)
else:
    print("❌ Cannot proceed - no functional datasets available for preprocessing")

### saving results report

In [ ]:
def save_preprocessing_summary_report(preprocessing_results, custom_outputdir):
    """
    Save a comprehensive preprocessing summary report to a text file
    
    Parameters:
    -----------
    preprocessing_results : dict
        Results from the preprocessing pipeline
    custom_outputdir : Path
        Output directory for saving the report
    """
    from datetime import datetime
    
    # Create report file path
    report_path = custom_outputdir / 'preprocessing_output' / 'PREPROCESSING_SUMMARY_REPORT.txt'
    report_path.parent.mkdir(parents=True, exist_ok=True)
    
    print(f"💾 Saving preprocessing summary report to: {report_path}")
    
    # Generate the summary data (same as in your print statements)
    summary_data = []
    for dataset_name, summary in preprocessing_results['processing_summary'].items():
        if 'error' not in summary:
            summary_data.append({
                'Dataset': dataset_name,
                'Original Features': summary['original_features'],
                'After Prevalence': summary['after_prevalence_filter'],
                'Final Features': summary['after_variance_filter'],
                'Total Removed': summary['total_features_removed'],
                'Removal %': f"{summary['total_removal_percentage']:.1f}%",
                'CLR Applied': 'Yes' if summary['clr_applied'] else 'No',
                'Status': 'Success' if summary['processing_successful'] else 'Failed'
            })
    
    # Write the report
    with open(report_path, 'w', encoding='utf-8') as f:
        # Header
        f.write("MICROBIOME FUNCTIONAL DATA PREPROCESSING SUMMARY REPORT\n")
        f.write("=" * 80 + "\n")
        f.write(f"Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write("=" * 80 + "\n\n")
        
        # Pipeline completion status
        f.write("🎉 PREPROCESSING PIPELINE COMPLETED FOR ALL DATASETS!\n")
        f.write("=" * 80 + "\n\n")
        
        # Detailed summary table
        f.write("📊 DETAILED PREPROCESSING SUMMARY:\n")
        f.write("-" * 80 + "\n\n")
        
        if summary_data:
            # Convert to DataFrame for nice formatting
            summary_df = pd.DataFrame(summary_data)
            f.write(summary_df.to_string(index=False))
            f.write("\n\n")
            
            # Calculate overall statistics
            total_original = sum([d['Original Features'] for d in summary_data])
            total_final = sum([int(d['Final Features']) for d in summary_data])
            overall_removal_pct = ((total_original - total_final) / total_original) * 100 if total_original > 0 else 0
            
            f.write("📈 OVERALL STATISTICS:\n")
            f.write("-" * 30 + "\n")
            f.write(f"   • Total original features: {total_original:,}\n")
            f.write(f"   • Total final features: {total_final:,}\n")
            f.write(f"   • Overall removal: {total_original - total_final:,} features ({overall_removal_pct:.1f}%)\n\n")
            
            # Individual dataset details
            f.write("📋 DATASET-SPECIFIC DETAILS:\n")
            f.write("-" * 40 + "\n")
            for data in summary_data:
                f.write(f"\n{data['Dataset']}:\n")
                f.write(f"  Original Features: {data['Original Features']:,}\n")
                f.write(f"  After Prevalence Filter: {data['After Prevalence']:,}\n")
                f.write(f"  Final Features: {data['Final Features']:,}\n")
                f.write(f"  Features Removed: {data['Total Removed']:,} ({data['Removal %']})\n")
                f.write(f"  CLR Transformation: {data['CLR Applied']}\n")
                f.write(f"  Processing Status: {data['Status']}\n")
        else:
            f.write("❌ No successful preprocessing results to summarize\n\n")
        
        # ENHANCED: Processing parameters with detailed methodology
        f.write("⚙️ PROCESSING PARAMETERS & METHODOLOGY:\n")
        f.write("=" * 50 + "\n\n")
        
        # Extract parameters from preprocessing results
        prevalence_threshold = 15.0  # Default values
        variance_percentile = 10.0
        epsilon_threshold = 1e-10
        
        # Try to extract actual parameters from results
        if 'filter_stats' in preprocessing_results:
            filter_stats = preprocessing_results['filter_stats']
            
            # Extract prevalence parameters
            if 'prevalence' in filter_stats:
                for dataset_name, stats in filter_stats['prevalence'].items():
                    if isinstance(stats, dict) and 'min_prevalence_threshold' in stats:
                        prevalence_threshold = stats['min_prevalence_threshold']
                        epsilon_threshold = stats.get('epsilon_threshold', 1e-10)
                        break
            
            # Extract variance parameters  
            if 'variance' in filter_stats:
                for dataset_name, stats in filter_stats['variance'].items():
                    if isinstance(stats, dict) and 'variance_percentile' in stats:
                        variance_percentile = stats['variance_percentile']
                        break
        
        # PREVALENCE FILTERING
        f.write("1. PREVALENCE FILTERING:\n")
        f.write("-" * 25 + "\n")
        f.write(f"   • Threshold: ≥{prevalence_threshold}% of samples\n")
        f.write(f"   • Epsilon threshold: {epsilon_threshold} (for PICRUSt2 small positive values)\n")
        f.write("   • Rationale: Remove features present in few samples to reduce noise\n")
        f.write("     and focus on consistently detected functions across the population.\n")
        f.write("   • Method: Features retained if present (> epsilon) in at least\n")
        f.write(f"     {prevalence_threshold}% of samples, accounting for PICRUSt2's small\n")
        f.write("     positive predictions rather than strict zero thresholding.\n\n")
        
        # VARIANCE FILTERING
        f.write("2. VARIANCE FILTERING:\n")
        f.write("-" * 20 + "\n")
        f.write(f"   • Threshold: ≥{variance_percentile}th percentile of Aitchison variance\n")
        f.write("   • Variance type: Aitchison variance (compositionally appropriate)\n")
        f.write("   • Rationale: Remove features with minimal variation across samples\n")
        f.write("     to focus on functionally dynamic features that differentiate samples.\n")
        f.write("   • Method: Aitchison variance calculated on CLR-transformed data,\n")
        f.write(f"     retaining features above the {variance_percentile}th percentile threshold.\n")
        f.write("     This accounts for the compositional nature of microbiome data.\n\n")
        
        # PSEUDOCOUNT METHODOLOGY
        f.write("3. PSEUDOCOUNT METHODOLOGY:\n")
        f.write("-" * 30 + "\n")
        f.write("   • Method: Quantile-bounded adaptive pseudocount\n")
        f.write("   • Calculation: 1st percentile of positive values / 2\n")
        f.write("   • Safety bounds: 1e-9 ≤ pseudocount ≤ 1e-3\n")
        f.write("   • Rationale: Handle zero values for CLR transformation while\n")
        f.write("     preserving the relative magnitude structure of the data.\n")
        f.write("   • Advantages over fixed pseudocount:\n")
        f.write("     - Adapts to each dataset's value distribution\n")
        f.write("     - Prevents artificial inflation of rare features\n")
        f.write("     - Maintains compositional relationships\n")
        f.write("     - Bounded to prevent numerical instability\n\n")
        
        # CLR TRANSFORMATION
        f.write("4. CLR TRANSFORMATION:\n")
        f.write("-" * 22 + "\n")
        f.write("   • Method: Centered Log-Ratio transformation\n")
        f.write("   • Formula: CLR(x) = log(x) - mean(log(x)) for each sample\n")
        f.write("   • Validation: Zero-sum property enforced (geometric mean = 0)\n")
        f.write("   • Rationale: Transform compositional data to real space,\n")
        f.write("     enabling standard statistical methods while preserving\n")
        f.write("     relative abundance information and sample-wise relationships.\n")
        f.write("   • Benefits:\n")
        f.write("     - Removes closure constraint of compositional data\n")
        f.write("     - Enables Euclidean-based statistics and ordinations\n")
        f.write("     - Preserves Aitchison geometry (appropriate for microbiome data)\n\n")
        
        # Extract and report actual pseudocount statistics if available
        if 'clr_stats' in preprocessing_results:
            f.write("5. PSEUDOCOUNT STATISTICS (ACTUAL VALUES USED):\n")
            f.write("-" * 50 + "\n")
            
            clr_stats = preprocessing_results['clr_stats']
            for dataset_name, stats in clr_stats.items():
                if isinstance(stats, dict) and 'pseudocount_stats' in stats:
                    ps_stats = stats['pseudocount_stats']
                    f.write(f"\n   {dataset_name}:\n")
                    f.write(f"     • Pseudocount value: {ps_stats.get('pseudocount_value', 'N/A'):.2e}\n")
                    f.write(f"     • Method: {ps_stats.get('method', 'N/A')}\n")
                    f.write(f"     • Values replaced: {ps_stats.get('zeros_replaced', 'N/A'):,} / {ps_stats.get('total_values', 'N/A'):,}")
                    f.write(f" ({ps_stats.get('replacement_percentage', 0):.1f}%)\n")
            f.write("\n")
        
        # Output directory information
        output_path = custom_outputdir / 'preprocessing_output'
        f.write("📁 OUTPUT LOCATIONS:\n")
        f.write("-" * 25 + "\n")
        f.write(f"   • Processed data: {output_path}\n")
        f.write(f"   • Summary plots: {output_path}\n")
        f.write(f"   • This report: {report_path}\n\n")
        
        # Analysis readiness section
        f.write("🔬 READY FOR DOWNSTREAM ANALYSIS!\n")
        f.write("-" * 40 + "\n")
        f.write("   • Filtered features by prevalence and variance using\n")
        f.write("     compositionally-appropriate methods\n")
        f.write("   • Applied robust quantile-bounded pseudocount strategy\n")
        f.write("   • CLR-transformed data ready for standard statistical methods\n")
        f.write("   • Maintained compositionality principles throughout pipeline\n")
        f.write("   • Type-safe preprocessing with comprehensive error handling\n")
        f.write("   • Compatible with PERMANOVA, PCoA, and multivariate statistics\n\n")
        
        # File listing (if available)
        if 'saved_files' in preprocessing_results:
            f.write("📄 GENERATED FILES:\n")
            f.write("-" * 25 + "\n")
            for file_key, file_path in preprocessing_results['saved_files'].items():
                f.write(f"   • {file_key}: {file_path}\n")
            f.write("\n")
        
        # METHODOLOGY REFERENCES
        f.write("📚 METHODOLOGICAL REFERENCES:\n")
        f.write("-" * 35 + "\n")
        f.write("   • Aitchison, J. (1986). The Statistical Analysis of Compositional Data.\n")
        f.write("   • Gloor, G.B., et al. (2017). Microbiome datasets are compositional.\n")
        f.write("   • Anderson, M.J. (2001). PERMANOVA for multivariate analysis.\n")
        f.write("   • Quinn, T.P., et al. (2018). Understanding sequencing data as\n")
        f.write("     compositions: An outlook and review. Bioinformatics.\n\n")
        
        # Footer
        f.write("=" * 80 + "\n")
        f.write("Report generated by Microbiome Functional Analysis Pipeline\n")
        f.write("Preprocessing designed for compositional microbiome data analysis\n")
        f.write("For questions or issues, please check the preprocessing code\n")
        f.write("Author: PhD. Guillermo G. Torres\n")
        f.write("=" * 80 + "\n")
    
    print(f"✅ Enhanced preprocessing summary report saved successfully!")
    return report_path

save_preprocessing_summary_report(preprocessing_results, custom_outputdir)

# 🧭 **Exploration & Structure Analysis**

## Multivariate Analysis of Functional Microbiome Data

This section performs comprehensive multivariate analysis of the CLR-transformed functional data:

### **Analysis Components:**
1. **Distance Matrices & Ordinations**: Aitchison distances (Euclidean on CLR-transformed data)
2. **PERMANOVA**: Permutational multivariate analysis of variance (accounting for KGcode blocking)
3. **PERMDISP**: Permutational analysis of multivariate dispersions (testing homogeneity of variances)
4. **PCoA Visualization**: Principal Coordinates Analysis with confidence ellipses and centroids

### **Statistical Framework:**
- **Distance Metric**: Aitchison distance (compositionally appropriate)
- **Ordination**: Principal Coordinates Analysis (PCoA)
- **Permutations**: 999 permutations for significance testing
- **Blocking**: Account for KGcode in PERMANOVA (equivalent to R's `strata` parameter)
- **Visualization**: PCoA scatter plots with stat_ellipse() equivalent

### Core functions

In [ ]:
"""
Comprehensive Multivariate Analysis Script for Microbiome Functional Data

This script performs:
1. Distance Matrices & Ordinations (Aitchison distances on CLR-transformed data)
2. PERMANOVA (with KGcode blocking)
3. PERMDISP (homogeneity of variances testing)
4. PCoA Visualization (with confidence ellipses and centroids)

Statistical Framework:
- Distance Metric: Aitchison distance (compositionally appropriate)
- Ordination: Principal Coordinates Analysis (PCoA)
- Permutations: 999 permutations for significance testing
- Blocking: Account for KGcode in PERMANOVA
- Visualization: PCoA scatter plots with confidence ellipses
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial.distance import pdist, squareform
from scipy.stats import f_oneway
from sklearn.metrics import pairwise_distances
import warnings
warnings.filterwarnings('ignore')

# Try to import skbio for better PERMANOVA implementation
try:
    from skbio.stats.distance import permanova
    SKBIO_AVAILABLE = True
    print("📦 scikit-bio available - using optimized PERMANOVA")
except ImportError:
    SKBIO_AVAILABLE = False
    print("⚠️  scikit-bio not available - using custom PERMANOVA implementation")

# Create output directory structure
def setup_output_directories(base_dir="multivariate_analysis"):
    """Setup organized directory structure for multivariate analysis"""
    base_path = Path(base_dir)
    
    # Create main directories
    dirs = {
        'base': base_path,
        'tables': base_path / 'tables',
        'graphs': base_path / 'graphs',
        'distance_matrices': base_path / 'tables' / 'distance_matrices',
        'pcoa_results': base_path / 'tables' / 'pcoa_results',
        'statistical_tests': base_path / 'tables' / 'statistical_tests',
        'ordination_plots': base_path / 'graphs' / 'ordination_plots',
    }
    
    # Create all directories
    for dir_path in dirs.values():
        dir_path.mkdir(parents=True, exist_ok=True)
    
    print(f"📁 Created directory structure in: {base_path.absolute()}")
    return dirs

# 1. DISTANCE MATRICES & ORDINATIONS
def calculate_aitchison_distance_matrix(data_df, feature_col=None, description_col='description'):
    """
    Calculate Aitchison distance matrix from CLR-transformed data
    
    Parameters:
    -----------
    data_df : pd.DataFrame
        CLR-transformed data with features and samples
    feature_col : str
        Column containing feature identifiers (e.g., 'function', 'pathway')
    description_col : str
        Column containing feature descriptions
        
    Returns:
    --------
    pd.DataFrame : Distance matrix with sample names as index/columns
    """
    print("Calculating Aitchison distance matrix...")
    
    # Identify abundance columns (exclude metadata columns)
    meta_cols = []
    if feature_col and feature_col in data_df.columns:
        meta_cols.append(feature_col)
    if description_col and description_col in data_df.columns:
        meta_cols.append(description_col)
    
    abundance_cols = [col for col in data_df.columns if col not in meta_cols]
    abundance_data = data_df[abundance_cols].copy()
    
    # Cast to numeric and handle NaN values
    abundance_data = abundance_data.apply(pd.to_numeric, errors='coerce')
    
    # Drop features (rows) that are all NaN
    abundance_data = abundance_data.dropna(how='all', axis=0)
    
    # Guard against degenerate matrices
    if abundance_data.shape[0] < 2:
        raise ValueError(f"Insufficient features for distance matrix calculation: "
                        f"only {abundance_data.shape[0]} features remaining after filtering")
    
    # Fill remaining NaN values with 0
    abundance_data = abundance_data.fillna(0.0)
    
    print(f"  Data shape for distance calculation: {abundance_data.shape}")
    print(f"  Features: {abundance_data.shape[0]}")
    print(f"  Samples: {len(abundance_cols)}")
    
    # Transpose so samples are rows for distance calculation
    sample_data = abundance_data.T
    
    # Calculate Euclidean distances (Aitchison distance on CLR data)
    distance_matrix = pairwise_distances(sample_data, metric='euclidean')
    
    # Return as DataFrame to maintain sample order
    distance_df = pd.DataFrame(
        distance_matrix,
        index=sample_data.index,
        columns=sample_data.index
    )
    
    print(f"  Distance matrix shape: {distance_df.shape}")
    return distance_df

def perform_pcoa(distance_matrix, n_components=10):
    """
    Perform Principal Coordinates Analysis (PCoA)
    
    Parameters:
    -----------
    distance_matrix : pd.DataFrame
        Square distance matrix
    n_components : int
        Number of components to compute
        
    Returns:
    --------
    dict : PCoA results with coordinates and variance explained
    """
    print("Performing Principal Coordinates Analysis (PCoA)...")
    
    # Convert to numpy array
    dist_array = distance_matrix.values
    
    # Double centering for PCoA
    n = dist_array.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    
    # Apply double centering: -0.5 * H * D² * H
    squared_distances = dist_array ** 2
    centered_matrix = -0.5 * H @ squared_distances @ H
    
    # Eigendecomposition
    eigenvals, eigenvecs = np.linalg.eigh(centered_matrix)
    
    # Sort by eigenvalues (descending)
    idx = np.argsort(eigenvals)[::-1]
    eigenvals = eigenvals[idx]
    eigenvecs = eigenvecs[:, idx]
    
    # Keep only positive eigenvalues and their corresponding eigenvectors
    positive_idx = eigenvals > 1e-8
    eigenvals_pos = eigenvals[positive_idx]
    eigenvecs_pos = eigenvecs[:, positive_idx]
    
    if len(eigenvals_pos) == 0:
        print("  Warning: No positive eigenvalues found")
        return {
            'coordinates': pd.DataFrame(),
            'eigenvalues': np.array([]),
            'variance_explained': np.array([]),
            'n_components': 0
        }
    
    # Calculate coordinates
    coordinates = eigenvecs_pos * np.sqrt(eigenvals_pos)
    
    # Limit to requested components
    n_comp = min(n_components, coordinates.shape[1])
    coordinates = coordinates[:, :n_comp]
    eigenvals_pos = eigenvals_pos[:n_comp]
    
    # Calculate variance explained - corrected to use only positive eigenvalues
    total_variance = eigenvals[eigenvals > 0].sum()
    variance_explained = eigenvals_pos / total_variance if total_variance > 0 else np.array([])
    
    # Create coordinates DataFrame
    coord_cols = [f'PC{i+1}' for i in range(n_comp)]
    coord_df = pd.DataFrame(
        coordinates,
        index=distance_matrix.index,
        columns=coord_cols
    )
    
    print(f"  PCoA completed: {n_comp} components extracted")
    if len(variance_explained) > 0:
        print(f"  PC1 variance explained: {variance_explained[0]:.3f}")
        if len(variance_explained) > 1:
            print(f"  PC2 variance explained: {variance_explained[1]:.3f}")
    
    return {
        'coordinates': coord_df,
        'eigenvalues': eigenvals_pos,
        'variance_explained': variance_explained,
        'n_components': n_comp
    }

# 2. PERMANOVA ANALYSIS
def permanova_test_skbio(distance_matrix, metadata_df, grouping_var, blocking_var=None, 
                        n_permutations=999, random_state=42):
    """
    Perform PERMANOVA test using scikit-bio (preferred method)
    """
    try:
        from skbio import DistanceMatrix
        from skbio.stats.distance import permanova
        
        # Align metadata to distance matrix order
        common_samples = [s for s in distance_matrix.index if s in metadata_df.index]
        
        if len(common_samples) < 3:
            print(f"  Error: Insufficient samples for PERMANOVA ({len(common_samples)} samples)")
            return {'error': 'Insufficient samples'}
        
        # Subset and align data
        dist_subset = distance_matrix.loc[common_samples, common_samples]
        meta_subset = metadata_df.loc[common_samples]
        
        # Get grouping information
        groups = meta_subset[grouping_var].dropna()
        unique_groups = groups.unique()
        
        if len(unique_groups) < 2:
            print(f"  Error: Need at least 2 groups for PERMANOVA")
            return {'error': 'Insufficient groups'}
        
        print(f"  Using scikit-bio PERMANOVA")
        print(f"  Samples: {len(common_samples)}")
        print(f"  Groups: {len(unique_groups)} ({', '.join(map(str, unique_groups))})")
        
        # Create scikit-bio DistanceMatrix
        skbio_dm = DistanceMatrix(dist_subset.values, ids=dist_subset.index)
        
        # Run PERMANOVA
        if blocking_var and blocking_var in meta_subset.columns:
            # With blocking (strata)
            strata = meta_subset[blocking_var]
            result = permanova(skbio_dm, groups, permutations=n_permutations, strata=strata)
        else:
            # Without blocking
            result = permanova(skbio_dm, groups, permutations=n_permutations)
        
        # Extract results
        results = {
            'f_statistic': result['test statistic'],
            'p_value': result['p-value'],
            'r_squared': result['test statistic'] / (result['test statistic'] + result['number of groups'] - 1),  # Approximate R²
            'n_permutations': result['number of permutations'],
            'n_samples': len(common_samples),
            'n_groups': len(unique_groups),
            'groups': unique_groups.tolist(),
            'blocking_used': blocking_var is not None,
            'method': 'scikit-bio'
        }
        
        print(f"  F-statistic: {results['f_statistic']:.4f}")
        print(f"  p-value: {results['p_value']:.4f}")
        print(f"  R²: {results['r_squared']:.4f}")
        
        return results
        
    except Exception as e:
        print(f"  Error with scikit-bio PERMANOVA: {e}")
        return {'error': str(e)}

def permanova_test_custom(distance_matrix, metadata_df, grouping_var, blocking_var=None, 
                         n_permutations=999, random_state=42):
    """
    Custom PERMANOVA implementation (fallback when scikit-bio unavailable)
    """
    print(f"  Using custom PERMANOVA implementation")
    
    # Create random number generator for thread safety
    rng = np.random.default_rng(random_state)
    
    # Align metadata to distance matrix order
    common_samples = [s for s in distance_matrix.index if s in metadata_df.index]
    
    if len(common_samples) < 3:
        print(f"  Error: Insufficient samples for PERMANOVA ({len(common_samples)} samples)")
        return {'error': 'Insufficient samples'}
    
    # Subset and align data
    dist_subset = distance_matrix.loc[common_samples, common_samples]
    meta_subset = metadata_df.loc[common_samples]
    
    # Get grouping information
    groups = meta_subset[grouping_var].dropna()
    unique_groups = groups.unique()
    
    if len(unique_groups) < 2:
        print(f"  Error: Need at least 2 groups for PERMANOVA")
        return {'error': 'Insufficient groups'}
    
    print(f"  Samples: {len(common_samples)}")
    print(f"  Groups: {len(unique_groups)} ({', '.join(map(str, unique_groups))})")
    
    # Calculate observed F-statistic (Anderson's method with centroid-based SS)
    def calculate_f_statistic_anderson(dist_mat, group_labels):
        """Calculate F-statistic using Anderson's centroid-based method"""
        n = len(group_labels)
        dist_array = dist_mat.values
        
        # Total sum of squared distances from grand centroid
        ss_total = np.sum(dist_array ** 2) / (2 * n)
        
        # Within-group sum of squares
        ss_within = 0
        df_within = 0
        
        for group in np.unique(group_labels):
            group_idx = np.where(group_labels == group)[0]
            n_group = len(group_idx)
            
            if n_group > 1:
                # Sum of squared distances within group
                group_dist = dist_array[np.ix_(group_idx, group_idx)]
                ss_within += np.sum(group_dist ** 2) / (2 * n_group)
                df_within += n_group - 1
        
        # Between-group sum of squares
        ss_between = ss_total - ss_within
        df_between = len(np.unique(group_labels)) - 1
        
        # F-statistic
        if df_within > 0:
            f_stat = (ss_between / df_between) / (ss_within / df_within)
        else:
            f_stat = np.inf
            
        return f_stat, ss_between, ss_within, ss_total
    
    # Calculate observed F-statistic
    obs_f, obs_ss_between, obs_ss_within, obs_ss_total = calculate_f_statistic_anderson(dist_subset, groups.values)
    
    # Permutation test
    f_permuted = []
    
    for i in range(n_permutations):
        # Permute group labels
        if blocking_var and blocking_var in meta_subset.columns:
            # Constrained permutation within blocks
            perm_groups = groups.copy()
            blocks = meta_subset[blocking_var]
            
            for block in blocks.unique():
                block_idx = blocks == block
                block_groups = groups[block_idx]
                if len(block_groups) > 1:
                    perm_groups[block_idx] = rng.permutation(block_groups.values)
        else:
            # Free permutation
            perm_groups = rng.permutation(groups.values)
        
        # Calculate F-statistic for permuted data
        perm_f, _, _, _ = calculate_f_statistic_anderson(dist_subset, perm_groups)
        f_permuted.append(perm_f)
    
    # Calculate p-value using (b + 1) / (m + 1) formula
    f_permuted = np.array(f_permuted)
    n_extreme = np.sum(f_permuted >= obs_f)
    p_value = (n_extreme + 1) / (n_permutations + 1)
    
    # Calculate R²
    r_squared = obs_ss_between / obs_ss_total if obs_ss_total > 0 else 0
    
    results = {
        'f_statistic': obs_f,
        'p_value': p_value,
        'r_squared': r_squared,
        'ss_between': obs_ss_between,
        'ss_within': obs_ss_within,
        'ss_total': obs_ss_total,
        'df_between': len(unique_groups) - 1,
        'df_within': len(common_samples) - len(unique_groups),
        'n_permutations': n_permutations,
        'n_samples': len(common_samples),
        'n_groups': len(unique_groups),
        'groups': unique_groups.tolist(),
        'blocking_used': blocking_var is not None,
        'method': 'custom'
    }
    
    print(f"  F-statistic: {obs_f:.4f}")
    print(f"  p-value: {p_value:.4f}")
    print(f"  R²: {r_squared:.4f}")
    
    return results

def permanova_test(distance_matrix, metadata_df, grouping_var, blocking_var=None, 
                  n_permutations=999, random_state=42):
    """
    Perform PERMANOVA test with optional blocking
    Uses scikit-bio if available, otherwise falls back to custom implementation
    """
    print(f"Performing PERMANOVA test (grouping: {grouping_var}, blocking: {blocking_var})...")
    
    if SKBIO_AVAILABLE:
        return permanova_test_skbio(distance_matrix, metadata_df, grouping_var, 
                                   blocking_var, n_permutations, random_state)
    else:
        return permanova_test_custom(distance_matrix, metadata_df, grouping_var, 
                                    blocking_var, n_permutations, random_state)

# 3. PERMDISP ANALYSIS
def permdisp_test(pcoa_coordinates, metadata_df, grouping_var, n_permutations=999, random_state=42):
    """
    Perform PERMDISP test using centroid-based distances in PCoA space
    
    Parameters:
    -----------
    pcoa_coordinates : pd.DataFrame
        PCoA coordinates
    metadata_df : pd.DataFrame
        Metadata with grouping variable
    grouping_var : str
        Column name for grouping variable
    n_permutations : int
        Number of permutations
    random_state : int
        Random seed for reproducibility
        
    Returns:
    --------
    dict : PERMDISP results
    """
    print(f"Performing PERMDISP test (grouping: {grouping_var})...")
    
    # Create random number generator for thread safety
    rng = np.random.default_rng(random_state)
    
    # Align data
    common_samples = [s for s in pcoa_coordinates.index if s in metadata_df.index]

    if len(common_samples) < 3:
        print(f"  Error: Insufficient samples for PERMDISP ({len(common_samples)} samples)")
        return {'error': 'Insufficient samples'}
    
    coords_subset = pcoa_coordinates.loc[common_samples]
    meta_subset = metadata_df.loc[common_samples]
    
    # Get grouping information
    groups = meta_subset[grouping_var].dropna()
    # IMPORTANT: Filter coordinates to match the non-NaN groups
    coords_subset = coords_subset.loc[groups.index]  # This ensures alignment
    
    unique_groups = groups.unique()
    
    if len(unique_groups) < 2:
        print(f"  Error: Need at least 2 groups for PERMDISP")
        return {'error': 'Insufficient groups'}
    
    print(f"  Using {coords_subset.shape[1]} PCoA axes")
    print(f"  Samples after removing NaN: {len(groups)}")
    print(f"  Groups: {len(unique_groups)} ({', '.join(map(str, unique_groups))})")
    
    # Calculate distances to group centroids
    def calculate_centroid_distances(coordinates, group_labels):
        distances_to_centroid = []
        group_dispersions = {}
        
        for group in np.unique(group_labels):
            group_idx = group_labels == group
            group_coords = coordinates[group_idx]
            
            if len(group_coords) > 0:
                # Calculate group centroid
                centroid = np.mean(group_coords, axis=0)
                
                # Calculate distances to centroid
                distances = np.sqrt(np.sum((group_coords - centroid) ** 2, axis=1))
                distances_to_centroid.extend(distances)
                
                group_dispersions[group] = {
                    'mean_distance': np.mean(distances),
                    'distances': distances
                }
        
        return np.array(distances_to_centroid), group_dispersions
    
    # Calculate observed dispersions
    obs_distances, obs_dispersions = calculate_centroid_distances(coords_subset.values, groups.values)
    
    # Test for homogeneity of dispersions using F-test
    group_mean_distances = [disp['mean_distance'] for disp in obs_dispersions.values()]
    group_distances_lists = [disp['distances'] for disp in obs_dispersions.values()]
    
    # Perform F-test between group dispersions
    if len(group_distances_lists) >= 2:
        f_stat, p_anova = f_oneway(*group_distances_lists)
    else:
        f_stat, p_anova = np.nan, np.nan
    
    # Permutation test for PERMDISP
    f_permuted = []
    
    for i in range(n_permutations):
        # Permute group labels using thread-safe RNG
        perm_groups = rng.permutation(groups.values)
        
        # Calculate dispersions for permuted groups
        _, perm_dispersions = calculate_centroid_distances(coords_subset.values, perm_groups)
        perm_distances_lists = [disp['distances'] for disp in perm_dispersions.values()]
        
        if len(perm_distances_lists) >= 2:
            perm_f, _ = f_oneway(*perm_distances_lists)
            f_permuted.append(perm_f)
    
    # Calculate permutation p-value using (b + 1) / (m + 1)
    if f_permuted:
        f_permuted = np.array(f_permuted)
        n_extreme = np.sum(f_permuted >= f_stat)
        p_perm = (n_extreme + 1) / (n_permutations + 1)
    else:
        p_perm = np.nan
    
    results = {
        'f_statistic': f_stat if not np.isnan(f_stat) else 0,
        'p_value_anova': p_anova if not np.isnan(p_anova) else 1,
        'p_value_permutation': p_perm if not np.isnan(p_perm) else 1,
        'group_dispersions': {str(group): float(disp['mean_distance']) 
                            for group, disp in obs_dispersions.items()},
        'n_permutations': n_permutations,
        'n_samples': len(common_samples),
        'n_groups': len(unique_groups),
        'groups': unique_groups.tolist()
    }
    
    print(f"  F-statistic: {f_stat:.4f}")
    print(f"  ANOVA p-value: {p_anova:.4f}")
    print(f"  Permutation p-value: {p_perm:.4f}")
    
    return results

# 4. VISUALIZATION FUNCTIONS
def plot_pcoa_with_ellipses(pcoa_results, metadata_df, grouping_var, 
                           colors=None, output_path=None, title_suffix=""):
    """
    Create PCoA plot with confidence ellipses and centroids
    Save in both PNG and PDF formats
    """
    
    # Default color palette if none provided
    if colors is None:
        colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#4CAF50', '#FF9800', '#9C27B0', '#673AB7']
    
    coordinates = pcoa_results['coordinates']
    variance_explained = pcoa_results['variance_explained']
    
    if coordinates.empty or len(variance_explained) < 2:
        print("  Warning: Insufficient data for PCoA plot")
        return None
    
    # Align data
    common_samples = [s for s in coordinates.index if s in metadata_df.index]
    coords_subset = coordinates.loc[common_samples]
    meta_subset = metadata_df.loc[common_samples]
    
    # Get grouping information
    groups = meta_subset[grouping_var].dropna()
    # IMPORTANT: Filter coordinates to match the non-NaN groups
    coords_subset = coords_subset.loc[groups.index]  # This ensures perfect alignment
    
    unique_groups = groups.unique()

    if len(unique_groups) < 1:
        print("  Warning: No valid groups found for plotting")
        return None
    
    print(f"  Plotting {len(groups)} samples in {len(unique_groups)} groups")
    
    # Create plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Plot points and ellipses for each group
    for i, group in enumerate(unique_groups):
        group_idx = groups == group
        group_coords = coords_subset[group_idx]
        
        if len(group_coords) == 0:
            continue
            
        color = colors[i % len(colors)]
        
        # Plot points
        ax.scatter(group_coords['PC1'], group_coords['PC2'], 
                  c=color, label=f'{group} (n={len(group_coords)})', 
                  alpha=0.7, s=60, edgecolor='white', linewidth=0.5)
        
        # Calculate and plot centroid
        centroid_x = group_coords['PC1'].mean()
        centroid_y = group_coords['PC2'].mean()
        ax.scatter(centroid_x, centroid_y, c=color, s=200, marker='x', 
                  linewidth=3, label=f'{group} centroid')
        
        # Plot confidence ellipse (95% confidence)
        if len(group_coords) > 2:
            from matplotlib.patches import Ellipse
            
            # Calculate covariance matrix
            coords_array = group_coords[['PC1', 'PC2']].values
            cov_matrix = np.cov(coords_array.T)
            
            # Calculate eigenvalues and eigenvectors
            eigenvals, eigenvecs = np.linalg.eigh(cov_matrix)
            
            # Calculate ellipse parameters
            angle = np.degrees(np.arctan2(eigenvecs[1, 1], eigenvecs[0, 1]))
            width = 2 * np.sqrt(5.991 * eigenvals[1])  # 95% confidence
            height = 2 * np.sqrt(5.991 * eigenvals[0])
            
            # Create and add ellipse
            ellipse = Ellipse((centroid_x, centroid_y), width, height, 
                            angle=angle, facecolor=color, alpha=0.2, 
                            edgecolor=color, linewidth=2)
            ax.add_patch(ellipse)
    
    # Customize plot
    pc1_var = variance_explained[0] * 100 if len(variance_explained) > 0 else 0
    pc2_var = variance_explained[1] * 100 if len(variance_explained) > 1 else 0
    
    ax.set_xlabel(f'PC1 ({pc1_var:.1f}% variance)')
    ax.set_ylabel(f'PC2 ({pc2_var:.1f}% variance)')
    ax.set_title(f'PCoA Plot - {grouping_var.replace("_", " ").title()}{title_suffix}')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='box')
    
    plt.tight_layout()
    
    # Save plot if path provided - save both PNG and PDF
    if output_path:
        output_path = Path(output_path)
        
        # Save PNG (high resolution)
        png_path = output_path.with_suffix('.png')
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        print(f"  PNG plot saved: {png_path}")
        
        # Save PDF (vector format)
        pdf_path = output_path.with_suffix('.pdf')
        plt.savefig(pdf_path, bbox_inches='tight')
        print(f"  PDF plot saved: {pdf_path}")
    
    plt.show()
    return fig

# 5. COMPREHENSIVE ANALYSIS FUNCTION
def comprehensive_multivariate_analysis(functional_data_dict, metadata_df, 
                                       grouping_vars, blocking_var='KGcode',
                                       output_dirs=None, n_permutations=999,
                                       random_state=42):
    """
    Perform comprehensive multivariate analysis on all functional datasets
    """
    
    print("\n" + "="*80)
    print("COMPREHENSIVE MULTIVARIATE ANALYSIS")
    print("="*80)
    
    all_results = {}
    
    # Process each functional dataset
    for dataset_name, data_df in functional_data_dict.items():
        print(f"\n{'='*20} ANALYZING {dataset_name.upper()} {'='*20}")
        
        try:
            # Determine feature column
            feature_col = 'function' if 'function' in data_df.columns else 'pathway'
            
            # 1. Calculate distance matrix
            print(f"\n1. Distance Matrix Calculation")
            distance_matrix = calculate_aitchison_distance_matrix(
                data_df, feature_col=feature_col
            )
            
            # Save distance matrix with row names
            if output_dirs:
                dist_path = output_dirs['distance_matrices'] / f"{dataset_name}_distance_matrix.csv"
                distance_matrix.to_csv(dist_path, index=True)  # Keep row names
                print(f"  Distance matrix saved: {dist_path}")
            
            # 2. Perform PCoA
            print(f"\n2. Principal Coordinates Analysis")
            pcoa_results = perform_pcoa(distance_matrix)
            
            # Save PCoA results with row names
            if output_dirs and not pcoa_results['coordinates'].empty:
                pcoa_path = output_dirs['pcoa_results'] / f"{dataset_name}_pcoa_coordinates.csv"
                pcoa_results['coordinates'].to_csv(pcoa_path, index=True)  # Keep row names (sample IDs)
                
                # Save eigenvalues and variance explained
                eigenvals_df = pd.DataFrame({
                    'PC': [f'PC{i+1}' for i in range(len(pcoa_results['eigenvalues']))],
                    'Eigenvalue': pcoa_results['eigenvalues'],
                    'Variance_Explained': pcoa_results['variance_explained']
                })
                eigenvals_path = output_dirs['pcoa_results'] / f"{dataset_name}_eigenvalues.csv"
                eigenvals_df.to_csv(eigenvals_path, index=False)
                
                print(f"  PCoA results saved: {pcoa_path}")
            
            # 3. Statistical tests for each grouping variable
            analysis_results = {
                'distance_matrix': distance_matrix,
                'pcoa_results': pcoa_results,
                'permanova_results': {},
                'permdisp_results': {}
            }
            
            for grouping_var in grouping_vars:
                if grouping_var not in metadata_df.columns:
                    print(f"  Warning: {grouping_var} not found in metadata")
                    continue
                
                print(f"\n3. Statistical Tests - {grouping_var}")
                
                # PERMANOVA
                permanova_result = permanova_test(
                    distance_matrix, metadata_df, grouping_var, 
                    blocking_var=blocking_var, n_permutations=n_permutations,
                    random_state=random_state
                )
                analysis_results['permanova_results'][grouping_var] = permanova_result
                
                # PERMDISP (if PCoA successful)
                if not pcoa_results['coordinates'].empty:
                    permdisp_result = permdisp_test(
                        pcoa_results['coordinates'], metadata_df, grouping_var,
                        n_permutations=n_permutations, random_state=random_state
                    )
                    analysis_results['permdisp_results'][grouping_var] = permdisp_result
                
                # Create visualization
                print(f"\n4. Visualization - {grouping_var}")
                if not pcoa_results['coordinates'].empty and len(pcoa_results['variance_explained']) >= 2:
                    plot_title = f" ({dataset_name})"
                    plot_path = None
                    if output_dirs:
                        plot_path = output_dirs['ordination_plots'] / f"{dataset_name}_{grouping_var}_pcoa"
                    
                    plot_pcoa_with_ellipses(
                        pcoa_results, metadata_df, grouping_var,
                        output_path=plot_path, title_suffix=plot_title
                    )
            
            # Save statistical results
            if output_dirs:
                # Save PERMANOVA results
                permanova_summary = []
                for var, result in analysis_results['permanova_results'].items():
                    if 'error' not in result:
                        permanova_summary.append({
                            'Dataset': dataset_name,
                            'Grouping_Variable': var,
                            'F_statistic': result['f_statistic'],
                            'p_value': result['p_value'],
                            'R_squared': result['r_squared'],
                            'n_samples': result['n_samples'],
                            'n_groups': result['n_groups'],
                            'blocking_used': result['blocking_used'],
                            'method': result.get('method', 'custom')
                        })
                
                if permanova_summary:
                    permanova_df = pd.DataFrame(permanova_summary)
                    permanova_path = output_dirs['statistical_tests'] / f"{dataset_name}_PERMANOVA_results.csv"
                    permanova_df.to_csv(permanova_path, index=False)
                    print(f"  PERMANOVA results saved: {permanova_path}")
                
                # Save PERMDISP results
                permdisp_summary = []
                for var, result in analysis_results['permdisp_results'].items():
                    if 'error' not in result:
                        permdisp_summary.append({
                            'Dataset': dataset_name,
                            'Grouping_Variable': var,
                            'F_statistic': result['f_statistic'],
                            'p_value_ANOVA': result['p_value_anova'],
                            'p_value_Permutation': result['p_value_permutation'],
                            'n_samples': result['n_samples'],
                            'n_groups': result['n_groups']
                        })
                
                if permdisp_summary:
                    permdisp_df = pd.DataFrame(permdisp_summary)
                    permdisp_path = output_dirs['statistical_tests'] / f"{dataset_name}_PERMDISP_results.csv"
                    permdisp_df.to_csv(permdisp_path, index=False)
                    print(f"  PERMDISP results saved: {permdisp_path}")
            
            all_results[dataset_name] = analysis_results
            
        except Exception as e:
            print(f"  Error analyzing {dataset_name}: {str(e)}")
            import traceback
            traceback.print_exc()
            all_results[dataset_name] = {'error': str(e)}
    
    return all_results

# 6. MAIN EXECUTION FUNCTION
def run_multivariate_analysis(preprocessing_results, metadata_df, 
                             grouping_vars=['Nutritionalstatus', 'infectioncoinfection', 'BH', 'GL', 'DF'],
                             blocking_var='KGcode', output_base_dir="multivariate_analysis"):
    """
    Main function to run complete multivariate analysis
    """
    
    # Setup output directories
    output_dirs = setup_output_directories(output_base_dir)
    
     # Extract CLR-transformed data -  variable initialization
    clr_data_dict = {}  # Initialize the variable first
    
    # Method 1: Check for direct dataset keys (original logic)
    for dataset_name, results in preprocessing_results.items():
        if isinstance(results, dict):
            if 'final_data' in results:
                clr_data_dict[dataset_name] = results['final_data']
            elif 'clr_transformed' in results:
                clr_data_dict[dataset_name] = results['clr_transformed']
    
    # Method 2: Check if 'clr_transformed' key contains the actual datasets
    if not clr_data_dict and 'clr_transformed' in preprocessing_results:
        clr_data = preprocessing_results['clr_transformed']
        
        if isinstance(clr_data, dict):
            # Multiple datasets in clr_transformed
            for dataset_name, dataset_data in clr_data.items():
                if hasattr(dataset_data, 'shape'):  # Check if it's DataFrame/array-like
                    clr_data_dict[dataset_name] = dataset_data
        elif hasattr(clr_data, 'shape'):
            # Single dataset in clr_transformed
            clr_data_dict['Dataset'] = clr_data
    
    # Method 3: Look for any DataFrame/array-like objects with 'clr' in the key name
    if not clr_data_dict:
        for key, value in preprocessing_results.items():
            if 'clr' in key.lower() and hasattr(value, 'shape'):
                clr_data_dict[key] = value
    
    if not clr_data_dict:
        print("Error: No CLR-transformed data found in preprocessing results")
        print(f"Available keys: {list(preprocessing_results.keys())}")
        if 'clr_transformed' in preprocessing_results:
            clr_item = preprocessing_results['clr_transformed']
            print(f"clr_transformed type: {type(clr_item)}")
            if isinstance(clr_item, dict):
                print(f"clr_transformed keys: {list(clr_item.keys())}")
        return None
    
    print(f"✅ Found CLR-transformed data for {len(clr_data_dict)} dataset(s):")
    for name, data in clr_data_dict.items():
        if hasattr(data, 'shape'):
            print(f"  - {name}: {data.shape}")
        else:
            print(f"  - {name}: {type(data)}")

    # Run comprehensive analysis
    analysis_results = comprehensive_multivariate_analysis(
        clr_data_dict, metadata_df, grouping_vars, 
        blocking_var=blocking_var, output_dirs=output_dirs
    )
    
    # Create summary report
    create_analysis_summary_report(analysis_results, output_dirs['base'])
    
    print(f"\n{'='*80}")
    print("✅ MULTIVARIATE ANALYSIS COMPLETED!")
    print(f"📁 Results saved in: {output_dirs['base'].absolute()}")
    print(f"{'='*80}")
    
    return analysis_results

def create_analysis_summary_report(analysis_results, output_dir):
    """Create a comprehensive summary report of all analyses"""
    
    report_path = output_dir / "MULTIVARIATE_ANALYSIS_SUMMARY.txt"
    
    with open(report_path, 'w') as f:
        f.write("MULTIVARIATE ANALYSIS SUMMARY REPORT\n")
        f.write("=" * 60 + "\n\n")
        
        for dataset_name, results in analysis_results.items():
            if 'error' in results:
                f.write(f"{dataset_name}: ERROR - {results['error']}\n\n")
                continue
                
            f.write(f"{dataset_name.upper()}\n")
            f.write("-" * 30 + "\n")
            
            # PCoA summary
            pcoa = results['pcoa_results']
            f.write(f"PCoA: {pcoa['n_components']} components extracted\n")
            if len(pcoa['variance_explained']) >= 2:
                f.write(f"  PC1: {pcoa['variance_explained'][0]:.3f} variance explained\n")
                f.write(f"  PC2: {pcoa['variance_explained'][1]:.3f} variance explained\n")
            
            # PERMANOVA results
            f.write("\nPERMANOVA Results:\n")
            for var, result in results['permanova_results'].items():
                if 'error' not in result:
                    f.write(f"  {var}:\n")
                    f.write(f"    F = {result['f_statistic']:.4f}\n")
                    f.write(f"    p = {result['p_value']:.4f}\n")
                    f.write(f"    R² = {result['r_squared']:.4f}\n")
                    f.write(f"    Method = {result.get('method', 'custom')}\n")
            
            # PERMDISP results  
            f.write("\nPERMDISP Results:\n")
            for var, result in results['permdisp_results'].items():
                if 'error' not in result:
                    f.write(f"  {var}:\n")
                    f.write(f"    F = {result['f_statistic']:.4f}\n")
                    f.write(f"    p (perm) = {result['p_value_permutation']:.4f}\n")
            
            f.write("\n" + "="*60 + "\n\n")
    
    print(f"Summary report saved: {report_path}")

# Example usage and testing
if __name__ == "__main__":
    print("Multivariate Analysis Script Loaded Successfully!")
    print(f"📦 scikit-bio available: {SKBIO_AVAILABLE}")
    print("\nMain functions available:")
    print("- setup_output_directories()")
    print("- calculate_aitchison_distance_matrix()")
    print("- perform_pcoa()")
    print("- permanova_test()")
    print("- permdisp_test()")
    print("- plot_pcoa_with_ellipses()")
    print("- comprehensive_multivariate_analysis()")
    print("- run_multivariate_analysis()")
    print("\nUse run_multivariate_analysis() to execute complete pipeline")

In [ ]:
multivardir = f"{custom_outputdir}/multivariate_analysis_results"
print(multivardir)

In [ ]:
# Execute comprehensive multivariate analysis
multivardir = f"{custom_outputdir}/multivariate_analysis_results"
# Check if we have the required data
if 'custom_outputdir' in globals() and 'preprocessing_results' in globals() and 'clean_data' in globals():
    
    try:
        # Prepare metadata
        if 'metadata' in clean_data:
            metadata_df = clean_data['metadata'].copy()
            
            # Set PC_code as index if it's not already
            if 'PC_code' in metadata_df.columns:
                metadata_df = metadata_df.set_index('PC_code')
            
            print("🧪 Checking available data...")
            print(f"📊 Preprocessing results: {list(preprocessing_results.keys())}")
            print(f"📋 Metadata shape: {metadata_df.shape}")
            print(f"📂 Custom directory: {custom_outputdir}")
            
            # Check what grouping variables are available
            potential_grouping_vars = ['Nutritionalstatus', 'infectioncoinfection', 'BH', 'GL', 'DF', 'HEI', 'AgeM', 'Sexchildren']
            available_vars = [var for var in potential_grouping_vars if var in metadata_df.columns]
            
            print(f"📈 Available grouping variables: {available_vars}")
            
            if available_vars:
                # Load and execute the multivariate analysis
                print("\n" + "="*60)
                # Run the analysis
                results = run_multivariate_analysis(
                    output_base_dir=multivardir,
                    preprocessing_results=preprocessing_results,
                    metadata_df=metadata_df,
                    grouping_vars=available_vars[:3],  # Test with first 3 variables
                    blocking_var='KGcode' if 'KGcode' in metadata_df.columns else None
                )
                if results:
                    print("\n✅ Multivariate analysis completed successfully!")
                    print(f"📁 Results saved in: {Path(multivardir) / 'multivariate_analysis'}")
                else:
                    print("\n❌ Multivariate analysis failed")
            else:
                print("❌ No suitable grouping variables found in metadata")
                print("Available columns:", list(metadata_df.columns))
        else:
            print("❌ No metadata found in clean_data")
            
    except Exception as e:
        print(f"❌ Error running multivariate analysis: {e}")
        import traceback
        traceback.print_exc()
        
else:
    print("⚠️  Required data not available:")
    if 'custom_outputdir' not in globals():
        print("  - custom_outputdir not defined")
    if 'preprocessing_results' not in globals():
        print("  - preprocessing_results not available")
    if 'clean_data' not in globals():
        print("  - clean_data not available")
    print("\nPlease run the preprocessing pipeline first")

## or just simply
#results = run_multivariate_analysis(
#    custom_dir=custom_outputdir,
#    preprocessing_results=preprocessing_results,
#    metadata_df=clean_data['metadata'].set_index('PC_code')
#)

###